<center><img src="https://raw.githubusercontent.com/mateuszk098/kaggle_notebooks/master/playground_series_s3e20/undraw_real_time_analytics_re_yliv.png" width=600px></center>

# <p style="font-family: 'JetBrains Mono'; font-weight: bold; font-size: 125%; color: #4A4B52; text-align: center">Playground Series S3E20 - CO2 Emission in Rwanda</p>


In [ ]:
# %load ../general_settings.py
!pip install -q -U swifter
import glob
import os
import shutil
import subprocess
import warnings
from array import array
from collections import defaultdict, namedtuple
from copy import copy
from functools import partial
from itertools import chain, combinations, product
from pathlib import Path
from time import strftime

ON_KAGGLE = os.getenv("KAGGLE_KERNEL_RUN_TYPE") is not None
if ON_KAGGLE:
    warnings.filterwarnings("ignore")

import joblib
import matplotlib.pyplot as plt
import numpy as np
import optuna
import pandas as pd
import plotly.express as px
import plotly.figure_factory as ff
import plotly.graph_objects as go
import scipy.stats as stats
import seaborn as sns
import shap
import swifter
from haversine import haversine
from colorama import Fore, Style
from IPython.core.display import HTML, display_html
from plotly.subplots import make_subplots
from scipy.cluster.hierarchy import linkage
from scipy.spatial.distance import squareform

from lightgbm import LGBMRegressor
from sklearn.model_selection import GroupKFold, cross_val_score
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import BaggingRegressor, RandomForestRegressor
from sklearn.model_selection import LeaveOneGroupOut
from sklearn.metrics import mean_squared_error
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.metrics import silhouette_score
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.utils.validation import check_is_fitted
from sklearn.cluster import KMeans
from sklearn.decomposition import TruncatedSVD
from sklearn.multioutput import MultiOutputRegressor
from sklearn.decomposition import NMF
from sklearn.metrics import explained_variance_score
from statsmodels.tsa.stattools import kpss, adfuller
from statsmodels.tsa.arima.model import ARIMA

# Colorama settings.
CLR = (Style.BRIGHT + Fore.BLACK) if ON_KAGGLE else (Style.BRIGHT + Fore.WHITE)
RED = Style.BRIGHT + Fore.RED
BLUE = Style.BRIGHT + Fore.BLUE
CYAN = Style.BRIGHT + Fore.CYAN
RESET = Style.RESET_ALL

FONT_COLOR = "#4A4B52"
BACKGROUND_COLOR = "#FFFCFA"

CELL_HOVER = {  # for row hover use <tr> instead of <td>
    "selector": "td:hover",
    "props": "background-color: #FFFCFA",
}
TEXT_HIGHLIGHT = {
    "selector": "td",
    "props": "color: #4A4B52; font-weight: bold",
}
INDEX_NAMES = {
    "selector": ".index_name",
    "props": "font-weight: normal; background-color: #FFFCFA; color: #4A4B52;",
}
HEADERS = {
    "selector": "th:not(.index_name)",
    "props": "font-weight: normal; background-color: #FFFCFA; color: #4A4B52;",
}
DF_STYLE = (INDEX_NAMES, HEADERS, TEXT_HIGHLIGHT)
DF_CMAP = sns.light_palette("#BAB8B8", as_cmap=True)

# Utility functions.
def download_from_kaggle(expr: list[str], directory: Path | None = None) -> None:
    if not directory:
        directory = Path("data")
    if not isinstance(directory, Path):
        raise TypeError("The `dir` argument must be `Path` instance!")
    match expr:
        case ["kaggle", _, "download", *args] if args:
            directory.parent.mkdir(parents=True, exist_ok=True)
            filename = args[-1].split("/")[-1] + ".zip"
            if not (directory / filename).is_file():
                subprocess.run(expr)
                shutil.unpack_archive(filename, directory)
                shutil.move(filename, directory)
        case _:
            raise SyntaxError("Invalid expression!")


def interpolate_color(color1, color2, t):
    r1, g1, b1 = int(color1[1:3], 16), int(color1[3:5], 16), int(color1[5:7], 16)
    r2, g2, b2 = int(color2[1:3], 16), int(color2[3:5], 16), int(color2[5:7], 16)
    r = int(r1 + (r2 - r1) * t)
    g = int(g1 + (g2 - g1) * t)
    b = int(b1 + (b2 - b1) * t)
    return f"#{r:02X}{g:02X}{b:02X}"


def get_interpolated_colors(color1, color2, num_colors=2):
    """Return `num_colors` interpolated beetwen `color1` and `color2`.
    Arguments need to be HEX."""
    num_colors = num_colors + 2
    return [interpolate_color(color1, color2, i / (num_colors - 1)) for i in range(num_colors)]


# Html `code` block highlight. Must be included at the end of all imports!
HTML(
    """
<style>
code {
    background: rgba(42, 53, 125, 0.10) !important;
    border-radius: 4px !important;
}
a {
    color: rgba(123, 171, 237, 1.0) !important;
}
ol.numbered-list {
  counter-reset: item;
}
ol.numbered-list li {
  display: block;
}
ol.numbered-list li:before {
  content: counters(item, '.') '. ';
  counter-increment: item;
}
</style>
"""
)


<p style="
    font-size: 20px;
    font-family: 'JetBrains Mono';
    color: #4A4B52;
">
    <b>Competition Description</b> 📜
</p>

<p style="
    font-size: 16px;
    font-family: 'JetBrains Mono';
    text-align: justify;
    text-justify: inter-word;
">
    The ability to accurately monitor carbon emissions is a critical step in the fight against climate change. Precise carbon readings allow researchers and governments to understand the sources and patterns of carbon mass output. While Europe and North America have extensive systems in place to monitor carbon emissions on the ground, there are few available in Africa. Approximately $497$ unique locations were selected from multiple areas in Rwanda, with a distribution around farm lands, cities and power plants. The data for this competition is split by time; the years $2019$ - $2021$ are included in the training data, and your task is to predict the CO2 emissions data for $2022$ through November.
</p>
    
<p style="
    font-size: 20px;
    font-family: 'JetBrains Mono';
    color: #4A4B52;
">
    <b>Task</b> 💡
</p>

<p style="
    font-size: 16px;
    font-family: 'JetBrains Mono';
    text-align: justify;
    text-justify: inter-word;
">
    The objective of this challenge is to create machine learning models that use open-source emissions data from <a href="https://sentinels.copernicus.eu/web/sentinel/missions/sentinel-5p"><b>Sentinel-5P</b></a> satellite observations to predict carbon emissions - <code>Emission</code>. These solutions may help enable governments, and other actors to estimate carbon emission levels across Africa, even in places where on-the-ground monitoring is not possible. The competition evaluation metric is <a href="https://en.wikipedia.org/wiki/Root-mean-square_deviation"><b>RMSE</b></a> (root mean squared error):
    \[\textrm{RMSE} = \sqrt{\frac{1}{N}\sum_{i=1}^N \left(y_i - \hat{y}_i\right)^2},\]
    where $y_i$ is the actual value and $\hat{y}_i$ is the predicted value, for the sample $i$.
</p>

<p style="
    font-size: 20px;
    font-family: 'JetBrains Mono';
    color: #4A4B52;
">
    <b>Table of Contents</b> 📔
</p>

<p style="
    font-size: 16px;
    font-family: 'JetBrains Mono';
    text-align: justify;
    text-justify: inter-word;
">
    The table of contents provides pleasurable navigation through the whole notebook. You can easily navigate through sections and return to TOC. If you want quickly find out something about the dataset, just read the first section, i.e. <b>Quick Overview</b>.
</p>

<blockquote class="anchor" id="top" style="
    margin-right: auto; 
    margin-left: auto;
    padding: 10px;
    background-color: #E04C5F;
    border-radius: 2px;
    border: 1px solid #E04C5F;
">
<ol class="numbered-list" style="
    font-size: 16px;
    font-family: 'JetBrains Mono';
    color: #F2F2F0;
    margin-top: 15px;
    margin-bottom: 15px;
">
    <li><a href="#quick_overview"><span style="color: #F2F2F0">Quick Overview</span></a>
    <ol class="numbered-list" class="numbered-list" style="
        font-size: 16px;
        font-family: 'JetBrains Mono';
        color: #F2F2F0;
    ">
        <li><a href="#data_reading_and_features_description"><span style="color: #F2F2F0">Data Reading &amp; Features Description</span></a></li>
        <li><a href="#samples_number_investigation_and_missing_values"><span style="color: #F2F2F0">Samples Number Investigation &amp; Missing Values</span></a></li>
        <li><a href="#emission_distribution_and_geospatial_locations"><span style="color: #F2F2F0">Emission Distribution &amp; Geospatial Locations</span></a></li>
        <li><a href="#time_series_quick_analysis"><span style="color: #F2F2F0">Time Series Quick Analysis</span></a></li>
        <li><a href="#quick_summary"><span style="color: #F2F2F0">Quick Summary</span></a></li>
    </ol>
    </li>
    <li><a href="#satellite_measurements_analysis"><span style="color: #F2F2F0">Satellite Measurements Analysis</span></a>
    <ol class="numbered-list" class="numbered-list" style="
        font-size: 16px;
        font-family: 'JetBrains Mono';
        color: #F2F2F0;
    ">
        <li><a href="#distributions"><span style="color: #F2F2F0">Distributions</span></a></li>
        <li><a href="#time_series"><span style="color: #F2F2F0">Time-Series</span></a></li>
    </ol>
    </li>
    <li><a href="#imputation_and_feature_engineering"><span style="color: #F2F2F0">Imputation &amp; Feature Engineering</span></a>
    <ol class="numbered-list" class="numbered-list" style="
        font-size: 16px;
        font-family: 'JetBrains Mono';
        color: #F2F2F0;
    ">
        <li><a href="#missing_values"><span style="color: #F2F2F0">Missing Values</span></a></li>
        <li><a href="#fixing_covid_period"><span style="color: #F2F2F0">Fixing Covid Period</span></a></li>
        <li><a href="#geospatial_clustering"><span style="color: #F2F2F0">Geospatial Clustering</span></a></li>
    </ol>
    </li>
    <li><a href="#machine_learning_model"><span style="color: #F2F2F0">Machine Learning Model</span></a>
    <ol class="numbered-list" class="numbered-list" style="
        font-size: 16px;
        font-family: 'JetBrains Mono';
        color: #F2F2F0;
    ">
        <li><a href="#basic_decision_tree"><span style="color: #F2F2F0">Basic Decision Tree</span></a></li>
        <li><a href="#better_approach_with_random_forest"><span style="color: #F2F2F0">Better Approach with Random Forest</span></a></li>
        <li><a href="#emission_decomposition_svd"><span style="color: #F2F2F0">Emission Decomposition - SVD</span></a></li>
        <li><a href="#emission_decomposition_nmf"><span style="color: #F2F2F0">Emission Decomposition - NMF</span></a></li>
        <li><a href="#nmf_approach_and_arima_models"><span style="color: #F2F2F0">NMF Approach &amp; ARIMA Models</span></a></li>
    </ol>
    </li>
    <li><a href="#summary"><span style="color: #F2F2F0">Summary</span></a>
</ol>
</blockquote>


# <b> <span style="font-family: 'JetBrains Mono'; color: #4A4B52">1</span> <span style='color: #E04C5F'>|</span> <span style="font-family: 'JetBrains Mono'; color: #4A4B52">Quick Overview</span></b><a class="anchor" id="quick_overview"></a> [↑](#top)


<p style="
    font-size: 20px;
    font-family: 'JetBrains Mono';
    color: #4A4B52;
    border-bottom: 2px solid #E04C5F;
">
    <b>About Section</b> 💡
</p>

<p style="
    font-size: 16px;
    font-family: 'JetBrains Mono';
    text-align: justify;
    text-justify: inter-word;
">
    In this section, I provide a quick overview of the dataset. More detailed analysis will be done in subsequent sections.
</p>

## <b> <span style="font-family: 'JetBrains Mono'; color: #4A4B52">1.1</span> <span style='color: #E04C5F'>|</span> <span style="font-family: 'JetBrains Mono'; color: #4A4B52">Data Reading &amp; Features Description</span></b><a class="anchor" id="data_reading_and_features_description"></a> [↑](#top)

In [ ]:
competition = "playground-series-s3e20"
expr = f"kaggle competitions download -c {competition}".split()

if not ON_KAGGLE:
    download_from_kaggle(expr)
    train_path = "data/train.csv"
    test_path = "data/test.csv"
else:
    train_path = f"/kaggle/input/{competition}/train.csv"
    test_path = f"/kaggle/input/{competition}/test.csv"

train = pd.read_csv(
    train_path,
    index_col="ID_LAT_LON_YEAR_WEEK",
    engine="pyarrow",
).rename(columns=str.title)

test = pd.read_csv(
    test_path,
    index_col="ID_LAT_LON_YEAR_WEEK",
    engine="pyarrow",
).rename(columns=str.title)


<p style="
    font-size: 20px;
    font-family: 'JetBrains Mono';
    color: #4A4B52;
    border-bottom: 2px solid #E04C5F;
">
    <b>Insight</b> 💡
</p>

<p style="
    font-size: 16px;
    font-family: 'JetBrains Mono';
    text-align: justify;
    text-justify: inter-word;
">
    I like to capitalize data frame columns because sometimes they can contain features named the same way as some methods. Since I prefer to refer to columns as an attribute (with a dot), it's better to provide capitalized names.
</p>

In [ ]:
train.head().style.set_table_styles(DF_STYLE)


<p style="
    font-size: 20px;
    font-family: 'JetBrains Mono';
    color: #4A4B52;
    border-bottom: 2px solid #E04C5F;
">
    <b>Features Description</b> 📔
</p>

<p style="
    font-size: 16px;
    font-family: 'JetBrains Mono';
    text-align: justify;
    text-justify: inter-word;
">
    Features are divided into seven major groups, and each such a group has several sub-features. Moreover, there we have those related to geographical location and date of collection, and obviously the target. 
</p>

<ul style="
    font-size: 16px;
    font-family: 'JetBrains Mono';
    text-align: justify;
    text-justify: inter-word;
">
    <li><code>Latitude</code> - Latitude of a sample collection.</li>
    <li><code>Longitude</code> - Longitude of a sample collection.</li>
    <li><code>Year</code> - Year of a sample collection.</li>
    <li><code>Week_No</code> - Week of a sample collection.</li>
    <li><code>SulphurDioxide-Variables</code> - Referring to: <a href="https://developers.google.com/earth-engine/datasets/catalog/COPERNICUS_S5P_NRTI_L3_SO2?hl=en"><b>Sentinel-5P NRTI SO2: Near Real-Time Sulfur Dioxide</b></a>. <i>Sulfur dioxide (SO2) enters the Earth's atmosphere through both natural and anthropogenic processes. It plays a role in chemistry on a local and global scale and its impact ranges from short-term pollution to effects on climate. Only about 30% of the emitted SO2 comes from natural sources; the majority is of anthropogenic origin. SO2 emissions adversely affect human health and air quality. SO2 has an effect on climate through radiative forcing, via the formation of sulfate aerosols. Volcanic SO2 emissions can also pose a threat to aviation, along with volcanic ash. S5P/TROPOMI samples the Earth's surface with a revisit time of one day with unprecedented spatial resolution of 3.5 x 7 km which allows the resolution of fine details including the detection of much smaller SO2 plumes.</i></li>
    <li><code>CarbonMonoxide-Variables</code> - Referring to: <a href="https://developers.google.com/earth-engine/datasets/catalog/COPERNICUS_S5P_NRTI_L3_CO?hl=en"><b>Sentinel-5P NRTI CO: Near Real-Time Carbon Monoxide</b></a>. <i>Carbon monoxide (CO) is an important atmospheric trace gas for understanding tropospheric chemistry. In certain urban areas, it is a major atmospheric pollutant. Main sources of CO are combustion of fossil fuels, biomass burning, and atmospheric oxidation of methane and other hydrocarbons. Whereas fossil fuel combustion is the main source of CO at northern mid-latitudes, the oxidation of isoprene and biomass burning play an important role in the tropics. TROPOMI on the Sentinel 5 Precursor (S5P) satellite observes the CO global abundance exploiting clear-sky and cloudy-sky Earth radiance measurements in the 2.3 μm spectral range of the shortwave infrared (SWIR) part of the solar spectrum. TROPOMI clear sky observations provide CO total columns with sensitivity to the tropospheric boundary layer. For cloudy atmospheres, the column sensitivity changes according to the light path.</i></li>
    <li><code>NitrogenDioxide-Variables</code> - Referring to: <a href="https://developers.google.com/earth-engine/datasets/catalog/COPERNICUS_S5P_NRTI_L3_NO2?hl=en"><b>Sentinel-5P NRTI NO2: Near Real-Time Nitrogen Dioxide</b></a>. <i>Nitrogen oxides (NO2 and NO) are important trace gases in the Earth's atmosphere, present in both the troposphere and the stratosphere. They enter the atmosphere as a result of anthropogenic activities (notably fossil fuel combustion and biomass burning) and natural processes (wildfires, lightning, and microbiological processes in soils). Here, NO2 is used to represent concentrations of collective nitrogen oxides because during daytime, i.e. in the presence of sunlight, a photochemical cycle involving ozone (O3) converts NO into NO2 and vice versa on a timescale of minutes. The TROPOMI NO2 processing system is based on the algorithm developments for the DOMINO-2 product and for the EU QA4ECV NO2 reprocessed dataset for OMI, and has been adapted for TROPOMI. This retrieval-assimilation-modelling system uses the 3-dimensional global TM5-MP chemistry transport model at a resolution of 1x1 degree as an essential element.</i></li>
    <li><code>Formaldehyde-Variables</code> - Referring to: <a href="https://developers.google.com/earth-engine/datasets/catalog/COPERNICUS_S5P_NRTI_L3_HCHO?hl=en"><b>Sentinel-5P NRTI HCHO: Near Real-Time Formaldehyde</b></a>. <i>Formaldehyde is an intermediate gas in almost all oxidation chains of non-methane volatile organic compounds (NMVOC), leading eventually to CO2. Non-Methane Volatile Organic Compounds (NMVOCs) are, together with NOx, CO and CH4, among the most important precursors of tropospheric O3. The major HCHO source in the remote atmosphere is CH4 oxidation. Over the continents, the oxidation of higher NMVOCs emitted from vegetation, fires, traffic and industrial sources results in important and localized enhancements of the HCHO levels. The seasonal and inter-annual variations of the formaldehyde distribution are principally related to temperature changes and fire events, but also to changes in anthropogenic activities. HCHO concentrations in the boundary layer can be directly related to the release of short-lived hydrocarbons, which mostly cannot be observed directly from space.</i></li>
    <li><code>UVAerosolIndex-Variables</code> - Referring to: <a href="https://developers.google.com/earth-engine/datasets/catalog/COPERNICUS_S5P_NRTI_L3_AER_AI?hl=en"><b>Sentinel-5P NRTI AER AI: Near Real-Time UV Aerosol Index</b></a>. <i>The AAI is based on wavelength-dependent changes in Rayleigh scattering in the UV spectral range for a pair of wavelengths. The difference between observed and modelled reflectance results in the AAI. When the AAI is positive, it indicates the presence of UV-absorbing aerosols like dust and smoke. It is useful for tracking the evolution of episodic aerosol plumes from dust outbreaks, volcanic ash, and biomass burning. The wavelengths used have very low ozone absorption, so unlike aerosol optical thickness measurements, AAI can be calculated in the presence of clouds. Daily global coverage is therefore possible.</i></li>
    <li><code>Ozone-Variables</code> - Referring to: <a href="https://developers.google.com/earth-engine/datasets/catalog/COPERNICUS_S5P_NRTI_L3_O3?hl=en"><b>Sentinel-5P NRTI O3: Near Real-Time Ozone</b></a>. <i>In the stratosphere, the ozone layer shields the biosphere from dangerous solar ultraviolet radiation. In the troposphere, it acts as an efficient cleansing agent, but at high concentration it also becomes harmful to the health of humans, animals, and vegetation. Ozone is also an important greenhouse-gas contributor to ongoing climate change. Since the discovery of the Antarctic ozone hole in the 1980s and the subsequent Montreal Protocol regulating the production of chlorine-containing ozone-depleting substances, ozone has been routinely monitored from the ground and from space.</i></li>
    <li><code>Cloud-Variables</code> - Referring to: <a href="https://developers.google.com/earth-engine/datasets/catalog/COPERNICUS_S5P_OFFL_L3_CLOUD?hl=en"><b>Sentinel-5P OFFL CLOUD: Near Real-Time Cloud</b></a>. <i>The TROPOMI/S5P cloud properties retrieval is based on the OCRA and ROCINN algorithms currently being used in the operational GOME and GOME-2 products. OCRA retrieves the cloud fraction using measurements in the UV/VIS spectral regions and ROCINN retrieves the cloud height (pressure) and optical thickness (albedo) using measurements in and around the oxygen A-band at 760 nm. Version 3.0 of the algorithms are used, which are based on a more realistic treatment of clouds as optically uniform layers of light-scattering particles. Additionally, the cloud parameters are also provided for a cloud model which assumes the cloud to be a Lambertian reflecting boundary.</i></li>
</ul>
<p style="
    font-size: 16px;
    font-family: 'JetBrains Mono';
    text-align: justify;
    text-justify: inter-word;
">
    Each of the above groups is divided by several sub-features, about whose you can read in the appropriate links. Moreover, there is another group - <code>UvAerosolLayerHeight</code>, but I didn't find any explanations about it. Moreover, as we see later, features related to that group are almost always missing. 
</p>

In [ ]:
train.info(verbose=False)

In [ ]:
test.info(verbose=False)

## <b> <span style="font-family: 'JetBrains Mono'; color: #4A4B52">1.2</span> <span style='color: #E04C5F'>|</span> <span style="font-family: 'JetBrains Mono'; color: #4A4B52">Samples Number Investigation &amp; Missing Values</span></b><a class="anchor" id="samples_number_investigation_and_missing_values"></a> [↑](#top)

<p style="
    font-size: 20px;
    font-family: 'JetBrains Mono';
    color: #4A4B52;
    border-bottom: 2px solid #E04C5F;
">
    <b>Insight</b> 💡
</p>

<p style="
    font-size: 16px;
    font-family: 'JetBrains Mono';
    text-align: justify;
    text-justify: inter-word;
">
    There we have the <code>Year</code> and <code>Week_No</code> of the observation collection, so it's fine to create a new column, i.e. <code>Date</code>, which allows us to easily navigate through the data frame. Moreover, we add the <code>Month_No</code> feature.    
</p>

In [ ]:
def extract_date_from_year_and_week(year, week):
    return pd.to_datetime(year, format="%Y") + pd.to_timedelta(week.mul(7), unit="days")


train["Date"] = extract_date_from_year_and_week(train.Year, train.Week_No)
test["Date"] = extract_date_from_year_and_week(test.Year, test.Week_No)

train["Month_No"] = train.Date.dt.month
test["Month_No"] = test.Date.dt.month


<p style="
    font-size: 20px;
    font-family: 'JetBrains Mono';
    color: #4A4B52;
    border-bottom: 2px solid #E04C5F;
">
    <b>Insight</b> 💡
</p>

<p style="
    font-size: 16px;
    font-family: 'JetBrains Mono';
    text-align: justify;
    text-justify: inter-word;
">
    Let's add the <code>Coordinates</code> feature too, in order to group and navigate through the data frame in an easy way.
</p>

In [ ]:
def get_coordinates(lat, lon):
    return "(" + lat.astype(str) + ", " + lon.astype(str) + ")"


train["Coordinates"] = get_coordinates(train.Latitude, train.Longitude)
test["Coordinates"] = get_coordinates(test.Latitude, test.Longitude)


In [ ]:
train_geo_pairs = train["Coordinates"].drop_duplicates().to_numpy()
test_geo_pairs = test["Coordinates"].drop_duplicates().to_numpy()

train_years = train.Year.unique()
test_years = test.Year.unique()

train_weeks = train.Week_No.unique()
test_weeks = test.Week_No.unique()

print(CLR + "Unique Geographical Locations in Train: ", RED + str(len(train_geo_pairs)))
print(CLR + "Unique Geographical Locations in Test:  ", RED + str(len(train_geo_pairs)), "\n")

print(CLR + "Years in Train:                        ", RED, *train_years)
print(CLR + "Years in Test:                         ", RED, *test_years, "\n")

print(CLR + "Weeks in Train:                        ", RED, train_weeks[0], "...", train_weeks[-1])
print(CLR + "Weeks in Test:                         ", RED, test_weeks[0], "...", test_weeks[-1])


In [ ]:
assert np.all(train_geo_pairs == test_geo_pairs)  # Are there the same locations?
assert len(train_geo_pairs) * len(train_years) * len(train_weeks) == len(train)
assert len(test_geo_pairs) * len(test_years) * len(test_weeks) == len(test)


<p style="
    font-size: 20px;
    font-family: 'JetBrains Mono';
    color: #4A4B52;
    border-bottom: 2px solid #E04C5F;
">
    <b>Geographical Locations and Date</b> 📜
</p>

<p style="
    font-size: 16px;
    font-family: 'JetBrains Mono';
    text-align: justify;
    text-justify: inter-word;
">
    Both training and test datasets have exactly $497$ distinct locations of sample collection, all of which are the same. The training dataset includes years $2019$, $2020$ and $2021$, and the test one, $2022$, but without December. <b>A simple calculation, i.e., $497$ (locations) * $3$ (years) * $53$ (number of weeks), gives us the length of the training dataset.</b><br><br>
    <b>Insight:</b> Perhaps it would be beneficial to provide a group cross-validation by year. Since we need to forecast almost the entire $2022$, we can evaluate the model on samples from previous years during the training.
</p>

In [ ]:
missing_values = pd.DataFrame(index=test.columns)
missing_values["MissingTrain"] = train.isna().sum()
missing_values["MissingTrainRatio"] = missing_values.MissingTrain / len(train)
missing_values["MissingTest"] = test.isna().sum()
missing_values["MissingTestRatio"] = missing_values.MissingTest / len(test)
missing_values.style.set_table_styles(DF_STYLE).background_gradient(DF_CMAP).set_precision(3)


<p style="
    font-size: 20px;
    font-family: 'JetBrains Mono';
    color: #4A4B52;
    border-bottom: 2px solid #E04C5F;
">
    <b>Missing Values</b> 📜
</p>

<p style="
    font-size: 16px;
    font-family: 'JetBrains Mono';
    text-align: justify;
    text-justify: inter-word;
">
    As you can see, we have many features, $74$ to be exact (I don't count <code>Date</code>, <code>Coordinates</code> and <code>Month_No</code>), and only $4$ of them have all the values. These are <code>Latitude</code>, <code>Longitude</code>, <code>Year</code>, and <code>Week_No</code>. As for the remaining, there are more or less missing values. The situation is the worst in the case of <code>UVAerosolLayerHeight-Variables</code>. Above $99$% of samples have missing values there regarding the training dataset. Taking this fact into account, we can reject these features.<br><br>
    <b>Insight:</b> Given the fact that geographical location is always known, there may be a method to utilise those to impute missing values. For example, when the satellite collect observations, these should be similar (in such way) for locations close to each other... Or maybe not. The second method may be to use the location and date of data collection.
</p>

In [ ]:
# We remove these features right away.
cols_to_reject = train.columns[train.columns.str.startswith("Uvaerosollayerheight")]

train = train.drop(cols_to_reject, axis=1)
test = test.drop(cols_to_reject, axis=1)


## <b> <span style="font-family: 'JetBrains Mono'; color: #4A4B52">1.3</span> <span style='color: #E04C5F'>|</span> <span style="font-family: 'JetBrains Mono'; color: #4A4B52">Emission Distribution &amp; Geospatial Locations</span></b><a class="anchor" id="emission_distribution_and_geospatial_locations"></a> [↑](#top)

In [ ]:
log_emission = np.log1p(train.Emission)
Q1, Q3 = np.percentile(log_emission, 25), np.percentile(log_emission, 75)

fig = px.histogram(
    x=log_emission,
    labels={"x": "Log(Emission)"},
    marginal="box",
    histnorm="probability density",
    title="CO\u2082 Emission Distribution after Logarithmic Transformation<br>"
    "<span style='font-size: 75%; font-weight: bold;'>"
    "In 2385 (3%) observations, zero-emission has been recorded</span>",
    color_discrete_sequence=["#4A4B52"],
    # opacity=0.75,
    nbins=100,
    height=540,
    width=840,
)
fig.add_vrect(x0=Q1, x1=Q3, line_width=0, fillcolor="#E04C5F", opacity=0.25, row=1)
fig.add_vline(x=Q1, line_width=2, line_dash="dash", line_color="#2D425E", row=1)
fig.add_vline(x=Q3, line_width=2, line_dash="dash", line_color="#2D425E", row=1)
fig.update_yaxes(title="Probability Density", row=1)
fig.update_layout(
    font_color=FONT_COLOR,
    title_font_size=18,
    plot_bgcolor=BACKGROUND_COLOR,
    paper_bgcolor=BACKGROUND_COLOR,
    xaxis_showgrid=False,
    yaxis_showgrid=False,
    bargap=0.25,
)
fig.show()


<p style="
    font-size: 20px;
    font-family: 'JetBrains Mono';
    color: #4A4B52;
    border-bottom: 2px solid #E04C5F;
">
    <b>Emission Distribution</b> 📜
</p>

<p style="
    font-size: 16px;
    font-family: 'JetBrains Mono';
    text-align: justify;
    text-justify: inter-word;
">
    The <code>Emission</code> feature (target) has a long tail, so it's more pleasurable to observe it after logarithmic transformation. In this case, we find out that <b>there are 2385 (3%) observations with zero-emission</b>. It's a little strange since there should be even a very small amount of emission. Perhaps observations haven't been collected, then? Let's have a look at samples with zero-emission more closely.
</p>

In [ ]:
mean_emission_by_loc = train.groupby("Coordinates").Emission.mean()
zero_emission_loc = mean_emission_by_loc[mean_emission_by_loc == 0]
print(
    CLR + "Always Zero-Emission Locations:\n\n",
    zero_emission_loc,
    CLR + "\n\nNumber of Locations with Always Zero-Emission: ",
    RED + f"{len(zero_emission_loc)}",
    sep="",
)


<p style="
    font-size: 20px;
    font-family: 'JetBrains Mono';
    color: #4A4B52;
    border-bottom: 2px solid #E04C5F;
">
    <b>Zero-Emission</b> 📜
</p>

<p style="
    font-size: 16px;
    font-family: 'JetBrains Mono';
    text-align: justify;
    text-justify: inter-word;
">
    In the training dataset, there we have $15$ distinct locations where zero-emission was always observed. Given the fact that such a pattern appeared in $2019$, $2020$, and $2021$, we could suppose that $2022$ remains at the same level. Well, it's time for a quick calculation. If there we have $15$ locations in the training dataset with zero-emission, $3$ different years and $53$ weeks, it gives us $15 * 3 * 53 = 2385$ observations. That is exactly the number with zero-emission observations. So for the test set, it would be $15 * 1 * 49 = 735$, which gives us $3$% of the whole test set. <b>We know $3$% of predictions for the test dataset. Of course, if the pattern is still there, which we don't know.</b> Now it's time to look at all coordinates on the map.
</p>

In [ ]:
geo_mean_emission = train.groupby(["Latitude", "Longitude"]).Emission.mean().reset_index()
zero_emission = geo_mean_emission[geo_mean_emission.Emission == 0]

fig = px.scatter_mapbox(
    geo_mean_emission,
    lat="Latitude",
    lon="Longitude",
    color="Emission",
    size="Emission",
    color_continuous_scale=px.colors.sequential.Cividis,
    size_max=30,
    zoom=7,
    width=840,
    height=940,
    title="Distinct Locations of Data Collection in Rwanda<br>"
    "<span style='font-size: 75%; font-weight: bold;'>"
    "Each dot is associated with the mean CO\u2082 emission collected for this place</span>",
)
fig.add_scattermapbox(
    lat=zero_emission.Latitude,
    lon=zero_emission.Longitude,
    name="Zero-Emission",
    marker=dict(color="#E04C5F", size=15, symbol="circle", opacity=0.75),
)
fig.update_layout(
    mapbox_style="open-street-map",
    margin=dict(r=0, t=90, l=0, b=0),
    font_color=FONT_COLOR,
    title_font_size=18,
    coloraxis_colorbar=dict(
        title="Mean Emission",
        title_side="top",
        orientation="h",
        yanchor="bottom",
        xanchor="center",
        y=-0.13,
        x=0.5,
    ),
    legend=dict(yanchor="bottom", xanchor="right", y=1, x=1, orientation="h"),
    plot_bgcolor=BACKGROUND_COLOR,
    paper_bgcolor=BACKGROUND_COLOR,
)
fig.show()


<p style="
    font-size: 20px;
    font-family: 'JetBrains Mono';
    color: #4A4B52;
    border-bottom: 2px solid #E04C5F;
">
    <b>Locations</b> 📜
</p>

<p style="
    font-size: 16px;
    font-family: 'JetBrains Mono';
    text-align: justify;
    text-justify: inter-word;
">
    Locations of sample collections are deployed around the whole of Rwanda. In this case, each dot is associated with the mean emission collected for this location <b>(except the zero-emission locations, for which I used the constant colour and size)</b>. At first sight, there are two special places which have much larger results of emission than the others. <b>These are: $(-2.378, 29.222)$ with the mean emission of $2233$, and $(-2.079, 29.321)$ with the mean emission of $1221$.</b> I checked those places with google maps, but there is nothing suspect, only some schools (?). Let's have a look at time series of those coordinates with comparison to the others.
</p>

## <b> <span style="font-family: 'JetBrains Mono'; color: #4A4B52">1.4</span> <span style='color: #E04C5F'>|</span> <span style="font-family: 'JetBrains Mono'; color: #4A4B52">Time Series Quick Analysis</span></b><a class="anchor" id="time_series_quick_analysis"></a> [↑](#top)

In [ ]:
two_highest_emissions_by_week = train[["Date", "Coordinates", "Emission"]].query(
    "(Coordinates == '(-2.378, 29.222)') | (Coordinates == '(-2.079, 29.321)')"
)
mean_emission_by_week = train.groupby("Date").Emission.mean().reset_index()

fig = px.line(
    two_highest_emissions_by_week,
    x="Date",
    y="Emission",
    line_dash="Coordinates",
    color="Coordinates",
    color_discrete_sequence=["#2D425E", "#3C5880"],
    title="Two Locations with the Highest CO\u2082 Emission vs Remaining Locations",
    height=480,
    width=840,
)
fig.update_traces(line_width=1.75, opacity=0.75)
fig.add_scatter(
    x=mean_emission_by_week.Date,
    y=mean_emission_by_week.Emission,
    name="Mean Emission",
    line=dict(width=1.75, color="#E04C5F", dash="dash"),
)
for coordinates in np.setdiff1d(
    train.Coordinates.unique(), ("(-2.378, 29.222)", "(-2.079, 29.321)")
):
    frame = train[train.Coordinates == coordinates]
    fig.add_scatter(
        x=frame.Date,
        y=frame.Emission,
        showlegend=False,
        name="",
        text=frame.Coordinates,
        hovertemplate="Coordinates=%{text}<br>Date=%{x}<br>Emission=%{y}",
        line=dict(width=0.25, color="rgba(68, 68, 68, 0.25)"),
    )
fig.update_yaxes(type="log", showgrid=False)
fig.update_xaxes(range=("2018-12-01", "2021-12-31"), showgrid=False)
fig.update_layout(
    font_color=FONT_COLOR,
    title_font_size=18,
    plot_bgcolor=BACKGROUND_COLOR,
    paper_bgcolor=BACKGROUND_COLOR,
    legend=dict(yanchor="bottom", xanchor="right", y=1, x=1, orientation="h", title=""),
)
fig.show()


<p style="
    font-size: 20px;
    font-family: 'JetBrains Mono';
    color: #4A4B52;
    border-bottom: 2px solid #E04C5F;
">
    <b>Emission Time Series</b> 📜
</p>

<p style="
    font-size: 16px;
    font-family: 'JetBrains Mono';
    text-align: justify;
    text-justify: inter-word;
">
    Well, as we can see, the process of emission is completely different for those two locations. For the former, there we have seasonal emission, which rapidly grows at the beginning of the year and last until the end of February. For the latter, the emission is rather similar during the whole period. <b>The remaining $495$ places have similar patterns, both in terms of values and seasonality.</b><br><br>
    <b>Insight:</b> Remember that the y-axis is in the logarithmic scale.
</p>

In [ ]:
mean_emission_per_week = (
    train.groupby("Week_No")
    .Emission.agg(
        EmissionMean="mean",
        CI=lambda x: stats.t.interval(0.95, len(x) - 1, loc=np.mean(x), scale=stats.sem(x)),
    )  # type: ignore
    .reset_index()
)
mean_emission_per_week[["CI_li", "CI_hi"]] = pd.DataFrame(
    mean_emission_per_week.CI.tolist(), index=mean_emission_per_week.index
)

fig = px.line(
    mean_emission_per_week,
    x="Week_No",
    y="EmissionMean",
    labels={"EmissionMean": "Mean Emission", "Week_No": "Week Number"},
    # markers=True,
    # symbol_sequence=["x-open"],
    color_discrete_sequence=["#3C5880"],
    title="Mean CO\u2082 Emission per Week - Including All Locations<br>"
    "<span style='font-size: 75%; font-weight: bold;'>"
    "The grey area shows the 95% confidence interval around the mean value</span>",
    height=480,
    width=840,
)
fig.update_traces(line_width=1.75, opacity=0.75)
fig.add_scatter(
    name="",
    x=mean_emission_per_week.Week_No,
    y=mean_emission_per_week.CI_hi,
    line_width=0,
    showlegend=False,
    hovertemplate="Week Number=%{x}<br>Higher Bound=%{y}",
)
fig.add_scatter(
    name="",
    x=mean_emission_per_week.Week_No,
    y=mean_emission_per_week.CI_li,
    line_width=0,
    showlegend=False,
    hovertemplate="Week Number=%{x}<br>Lower Bound=%{y}",
    fillcolor="rgba(68, 68, 68, 0.15)",
    fill="tonexty",
)
fig.add_annotation(
    x=2,
    y=35,
    align="left",
    xanchor="left",
    text="<b>Mean emission is, on average higher by 31% for weeks 14, 15 and 16 (April).</b><br>"
    "<b>Mean emission is, on average higher by 33% for weeks 40, 41, 42 and 43 (October).</b><br><br>"
    "<b>Rwanda characterises rainy seasons from March to May and from September to November.<b><br>"
    "<b>Correlation?</b>",
    showarrow=False,
)
fig.update_layout(
    font_color=FONT_COLOR,
    title_font_size=18,
    plot_bgcolor=BACKGROUND_COLOR,
    paper_bgcolor=BACKGROUND_COLOR,
    xaxis_showgrid=False,
    yaxis_showgrid=False,
    xaxis_range=(-1, 53),
    yaxis_range=(0, 140),
    legend=dict(yanchor="bottom", xanchor="right", y=1, x=1, orientation="h", title=""),
)
fig.show()


In [ ]:
mean_emission_per_year_and_week = train.groupby(["Year", "Week_No"]).Emission.mean().reset_index()

fig = px.line(
    mean_emission_per_year_and_week,
    x="Week_No",
    y="Emission",
    labels={"Emission": "Mean Emission", "Week_No": "Week Number"},
    color="Year",
    color_discrete_sequence=["#6B6A6A", "#E04C5F", "#6B6A6A"],
    line_dash="Year",
    line_dash_sequence=["dashdot", "solid", "dash"],
    title="Mean CO\u2082 Emission per Week & Year - Including All Locations",
    height=480,
    width=840,
)
fig.update_traces(line_width=1.75, opacity=0.75)
fig.add_annotation(
    x=2,
    y=25,
    align="left",
    xanchor="left",
    text="<b>On 14 March 2020, the Government of Rwanda closed all schools in the country</b><br>"
    "<b>due to the COVID-19 pandemic. Since then, we notice a drop in the mean emission.</b><br>"
    "<b>At the end of the year, everything seems to get back to normal.</b>",
    showarrow=False,
)
fig.update_layout(
    font_color=FONT_COLOR,
    title_font_size=18,
    plot_bgcolor=BACKGROUND_COLOR,
    paper_bgcolor=BACKGROUND_COLOR,
    xaxis_showgrid=False,
    yaxis_showgrid=False,
    xaxis_range=(-1, 53),
    yaxis_range=(0, 140),
    legend=dict(yanchor="bottom", xanchor="right", y=1, x=1, orientation="h", title=""),
)
fig.show()


<p style="
    font-size: 20px;
    font-family: 'JetBrains Mono';
    color: #4A4B52;
    border-bottom: 2px solid #E04C5F;
">
    <b>Weekly Emission</b> 📜
</p>

<p style="
    font-size: 16px;
    font-family: 'JetBrains Mono';
    text-align: justify;
    text-justify: inter-word;
">
    First plot: <b>April and October seem to have a significantly higher level of emission than the other months.</b> Why is that? Unfortunately, it's hard to say now, and this requires further investigation, perhaps. Nevertheless, here is a link to the topic about that by <a href="https://www.kaggle.com/nivedithavudayagiri"><b>Niveditha Vudayagiri</b></a>: <a href="https://www.kaggle.com/competitions/playground-series-s3e20/discussion/428540"><b>Higher emissions in May and October?</b></a><br><br>
    Second plot: <b>We observe a decrease in the mean CO2 emission since the Covid-19 outbreak in Rwanda in March $2020$. On $14$ March $2020$, the Government of Rwanda closed all schools in the country. In the third quarter of $2020$, the situation begins to stabilise.</b><br><br>
    <b>Insight:</b> We could shift a little the emission data during Covid-19 since we know that it's a particular case and doesn't have an impact on $2022$ (the year we try to forecast). We could, for example, multiply these values by some factors determined using the years $2019$ and $2021$.<br><br>
    Let's have a look at the quarter-over-quarter relative difference in mean emission for each geospatial location.
</p>

In [ ]:
def symlog_transform(x):
    return np.sign(x) * np.log10(1 + np.abs(x))


locs_by_year_q = (
    train.assign(Quarter=train.Date.dt.quarter.replace({1: "Q1", 2: "Q2", 3: "Q3", 4: "Q4"}))
    .groupby(["Coordinates", "Year", "Quarter"])
    .Emission.mean()
    .reset_index()
)
quarter_over_quarter_frame = (
    locs_by_year_q.groupby("Coordinates")
    .Emission.diff(1)  # Quarter-over-quarter difference.
    .fillna(0)  # Treat 2019-Q1 as a baseline - 0.
    .to_frame("Emission Relative Difference")
    .join(locs_by_year_q)
    .assign(YearQuarter=locs_by_year_q.Quarter + "'" + locs_by_year_q.Year.astype(str).str[-2:])
)

fig = px.line(
    quarter_over_quarter_frame,
    x="YearQuarter",
    y=symlog_transform(quarter_over_quarter_frame["Emission Relative Difference"]),
    labels={"y": "Emission Relative Difference", "YearQuarter": "Date"},
    color="Coordinates",
    color_discrete_sequence=["rgba(68, 68, 68, 0.15)"],
    hover_data="Emission Relative Difference",
    title="Quarter-over-Quarter Mean CO\u2082 Emission Difference<br>"
    "<span style='font-size: 75%; font-weight: bold;'>"
    "Each line is associated with a specific geospatial location</span>",
    height=480,
    width=840,
)
fig.update_traces(line_width=0.75)
fig.update_yaxes(
    tickvals=np.arange(-3, 4, 1), ticktext=["-1000", "-100", "-10", "0", "10", "100", "1000"]
)
fig.update_layout(
    font_color=FONT_COLOR,
    title_font_size=18,
    plot_bgcolor=BACKGROUND_COLOR,
    paper_bgcolor=BACKGROUND_COLOR,
    showlegend=False,
    xaxis_showgrid=False,
    yaxis_showgrid=False,
)
fig.show()


<p style="
    font-size: 20px;
    font-family: 'JetBrains Mono';
    color: #4A4B52;
    border-bottom: 2px solid #E04C5F;
">
    <b>Quarter-over-Quarter Differences</b> 📜
</p>

<p style="
    font-size: 16px;
    font-family: 'JetBrains Mono';
    text-align: justify;
    text-justify: inter-word;
">
    Here we have a kind of interesting plot for each geospatial location in the training dataset. We computed a mean emission in the first quarter of $2019$ for each location, and as these values depict our baseline for that time, the relative difference equals zero. So, in the second quarter of $2019$, emissions rather tend to increase (but not for all locations). Subsequently, these drop in Q3 compared to Q2 and grow again in Q4 in regards to Q3. We saw such a pattern in earlier plots. Notice that in $2020$ there is a little mess, perhaps related to Covid-19 pandemic.
</p>

In [ ]:
year_over_year_change = (
    quarter_over_quarter_frame.groupby(["Year", "Quarter"])
    .Emission.mean()
    .pct_change(4)  # Percentage change between this quarter and quarter year before.
    .fillna(0)  # Treat 2019 as a baseline, thus assign zeros.
    .mul(100)  # To percent.
    .round(0)
    .astype(np.int8)
    .reset_index()
    .assign(YearQuarter=locs_by_year_q.Quarter + "'" + locs_by_year_q.Year.astype(str).str[-2:])
    .iloc[-8:]  # 2020 and 2021.
)

fig = px.line(
    year_over_year_change,
    x="YearQuarter",
    y="Emission",
    labels={"Emission": "Year-over-Year Percentage Change", "YearQuarter": "Date"},
    markers=True,
    symbol_sequence=["circle"],
    text="Emission",
    color_discrete_sequence=["#3C5880"],
    title="Year-over-Year Percentage Change in Mean CO\u2082 Emission<br>"
    "<span style='font-size: 75%; font-weight: bold;'>"
    "In 2021 there is a significant increase compared to the corresponding quarters in 2020</span>",
    height=480,
    width=840,
)
fig.add_hline(y=0, line_width=2, line_dash="dash", line_color="rgba(68, 68, 68, 0.25)")
fig.update_traces(
    line_width=1.75,
    opacity=0.75,
    marker_size=8,
    marker_color=BACKGROUND_COLOR,
    marker_line_width=2,
    textposition=["bottom left"] * 2 + ["bottom right"] * 3 + ["top right"] * 3,
    textfont_size=12,
    textfont_color="#2D425E",
    textfont_family="Arial Black",
    texttemplate="%{y:+}",
)
fig.add_annotation(
    x=5.3,
    y=-20,
    align="left",
    xanchor="left",
    text="<b>Will the trend continue?</b>",
    showarrow=False,
)
fig.add_annotation(
    x=-0.1,
    y=28,
    align="left",
    xanchor="left",
    text="<b>Each point is related to a change in comparison</b><br>"
    "<b>to the analogous quarter in the previous year.</b>",
    showarrow=False,
)
fig.update_layout(
    font_color=FONT_COLOR,
    title_font_size=18,
    plot_bgcolor=BACKGROUND_COLOR,
    paper_bgcolor=BACKGROUND_COLOR,
    xaxis_showgrid=False,
    yaxis_showgrid=False,
    yaxis_range=(-40, 40),
    legend=dict(yanchor="bottom", xanchor="right", y=1, x=1, orientation="h", title=""),
)
fig.show()


<p style="
    font-size: 20px;
    font-family: 'JetBrains Mono';
    color: #4A4B52;
    border-bottom: 2px solid #E04C5F;
">
    <b>Year-over-Year Percentage Change</b> 📜
</p>

<p style="
    font-size: 16px;
    font-family: 'JetBrains Mono';
    text-align: justify;
    text-justify: inter-word;
">
    The plot above depicts year-over-year percentage changes in mean CO2 emission for all locations. There is clearly visible drop in $2020$ in comparison to $2019$, but in $2021$ emission begins to grow. <b>Is this growth will be still there in $2022$?</b>
</p>

## <b> <span style="font-family: 'JetBrains Mono'; color: #4A4B52">1.5</span> <span style='color: #E04C5F'>|</span> <span style="font-family: 'JetBrains Mono'; color: #4A4B52">Quick Summary</span></b><a class="anchor" id="quick_summary"></a> [↑](#top)

<p style="
    font-size: 20px;
    font-family: 'JetBrains Mono';
    color: #4A4B52;
    border-bottom: 2px solid #E04C5F;
">
    <b>What We Already Know</b> 📔
</p>

<p style="
    font-size: 16px;
    font-family: 'JetBrains Mono';
    text-align: justify;
    text-justify: inter-word;
">
    <b>The first section provides a quick overview of the available dataset. In this section we found out several things:</b>
</p>

<ul style="
    font-size: 16px;
    font-family: 'JetBrains Mono';
    text-align: justify;
    text-justify: inter-word;
">
    <li>This competition is a time series regression one with the <b>RMSE</b> metric.</li>
    <li>The training dataset contains $74$ features, and only four of them don't have any missing values. These are associated with the geographical location of the observation collection and its date:
    <ul style="
        font-size: 16px;
        font-family: 'JetBrains Mono';
    ">
        <li><code>Latitude</code>,</li>
        <li><code>Longitude</code>,</li>
        <li><code>Year</code>,</li>
        <li><code>Week_No</code>.</li>
    </ul>
    </li>
    <li>Features are related to seven different groups of measurements collected by the Sentinel-5P satellite. These groups are divided into sub-features related, for example, to the zenith angle of the satellite at the ground pixel location or altitude of the satellite with respect to the geodetic sub-satellite point.</li>
    <li>Observations have been collected from $497$ distinct geographical locations. All of these locations appear in both the training and the test dataset. Samples in the training dataset include years $2019$, $2020$ and $2021$, and in the test dataset a year $2022$ (without December).</li>
    <li>The training dataset is composed of $497 * 3  * 53 = 79023$ observations, meanwhile, the test one is composed of $497 * 1  * 49 = 24353$ samples.</li>
    <li>Distribution of <code>Emission</code> (target to forecast) has a long tail, due to two locations with enormously higher emissions than the remaining $495$ places. These are $(-2.378, 29.222)$ and $(-2.079, 29.321)$.</li>
    <li>The following locations: $(-2.378, 29.222)$ and $(-2.079, 29.321)$ (with significantly higher emission) have completely different patterns in time than the remaining ones. The first has more or less constant emission during the year, meanwhile the second show strong seasonality, where emission rapidly grows at the beginning of the year and such a pattern last until the end of February.</li>
    <li>In the training dataset, there we have $15$ different locations, where always zero-emission was recorded, which gives us $2385$ observations in total. Suppose that such behaviour is constant we can forecast emissions for those places at hand (just zero), which will give us $15 * 1 * 49 = 735$ known predictions ($3$% of the test dataset).</li>
    <li>Places of sample collection are deployed around the whole of Rwanda, and many of them have different levels of mean emission despite being located close to each other.</li>
    <li>Mean emission per week has similar values across the year, except for April and October months, where rapidly grows by around 30% compared to the mean value.</li>
    <li>The fact that all locations are always known and there are missing values related to them, those can (probably) be imputed considering nearby places. Moreover, imputation should be done with regard to the date of measurement collection.</li>
    <li>Emissions usually grow in the second quarter of the year, as for to drop in the third quarter and rise again in the last quarter. The pattern seems to be broken in $2020$, probably due to the Covid-19 pandemic.</li>
</ul>

<p style="
    font-size: 16px;
    font-family: 'JetBrains Mono';
    text-align: justify;
    text-justify: inter-word;
">
    <b>We know already more or less about the dataset, but we don't know anything about measurements from the Sentinel-5P satellite. This would be a topic of the subsequent sections.</b>
</p>

# <b> <span style="font-family: 'JetBrains Mono'; color: #4A4B52">2</span> <span style='color: #E04C5F'>|</span> <span style="font-family: 'JetBrains Mono'; color: #4A4B52">Satellite Measurements Analysis</span></b><a class="anchor" id="satellite_measurements_analysis"></a> [↑](#top)

<p style="
    font-size: 20px;
    font-family: 'JetBrains Mono';
    color: #4A4B52;
    border-bottom: 2px solid #E04C5F;
">
    <b>About Section</b> 💡
</p>

<p style="
    font-size: 16px;
    font-family: 'JetBrains Mono';
    text-align: justify;
    text-justify: inter-word;
">
    In this section, we analyse features related to satellite measurements. For this case, we draw distributions of these measurements for both, training and test datasets and compare differences. Moreover, we discover an interesting finding. Subsequently, we depict time-series plots of these measurements.
</p>

## <b> <span style="font-family: 'JetBrains Mono'; color: #4A4B52">2.1</span> <span style='color: #E04C5F'>|</span> <span style="font-family: 'JetBrains Mono'; color: #4A4B52">Distributions</span></b><a class="anchor" id="distributions"></a> [↑](#top)

<p style="
    font-size: 16px;
    font-family: 'JetBrains Mono';
">
    <b>Let's get started with distributions of satellite measurements.</b>
</p>

In [ ]:
def get_n_rows_axes(n_features, n_cols):
    n_rows = int(np.ceil(n_features / n_cols))
    current_col = range(1, n_cols + 1)
    current_row = range(1, n_rows + 1)
    return n_rows, list(product(current_row, current_col))


def draw_dist_for_group(group, vars_to_plot, n_cols=3, height=640):
    n_rows, axes = get_n_rows_axes(len(vars_to_plot), n_cols)
    fig = make_subplots(
        rows=n_rows,
        cols=n_cols,
        y_title="Probability Density",
        horizontal_spacing=0.1,
        vertical_spacing=0.1,
    )
    fig.update_annotations(font_size=14)
    for frame, color, frame_name in zip((train, test), ("#3C5880", "#E04C5F"), ("Train", "Test")):
        for k, (var, (row, col)) in enumerate(zip(vars_to_plot, axes), start=1):
            density, bins = np.histogram(frame[var].dropna(), bins=200, density=True)
            fig.add_bar(
                x=bins,
                y=density,
                marker_color=color,
                marker_line_width=0,
                # marker_line_color=color,
                opacity=0.5,
                name=frame_name,
                legendgroup=frame_name,
                showlegend=k == 1,
                row=row,
                col=col,
            )
            fig.update_xaxes(
                tickfont_size=7,
                showgrid=False,
                title_text=var,
                titlefont_size=7,
                titlefont_family="Arial Black",
                row=row,
                col=col,
            )
            fig.update_yaxes(tickfont_size=7, showgrid=False, row=row, col=col)

    fig.update_layout(
        width=840,
        height=height,
        title=f"{group} Features - Probability Density",
        font_color=FONT_COLOR,
        title_font_size=18,
        plot_bgcolor=BACKGROUND_COLOR,
        paper_bgcolor=BACKGROUND_COLOR,
        bargap=0,
        legend=dict(yanchor="bottom", xanchor="right", y=1, x=1, orientation="h", title=""),
    )
    return fig


sulphurdioxide_group = "Sulphurdioxide"
sulphurdioxide_vars = train.columns[train.columns.str.startswith(sulphurdioxide_group)]

fig = draw_dist_for_group(sulphurdioxide_group, sulphurdioxide_vars)
fig.show()


In [ ]:
carbonmonoxide_group = "Carbonmonoxide"
carbonmonoxide_vars = train.columns[train.columns.str.startswith(carbonmonoxide_group)]
fig = draw_dist_for_group(carbonmonoxide_group, carbonmonoxide_vars)
fig.show()

In [ ]:
nitrogendioxide_group = "Nitrogendioxide"
nitrogendioxide_vars = train.columns[train.columns.str.startswith(nitrogendioxide_group)]
fig = draw_dist_for_group(nitrogendioxide_group, nitrogendioxide_vars, height=740)
fig.show()

In [ ]:
formaldehyde_group = "Formaldehyde"
formaldehyde_vars = train.columns[train.columns.str.startswith(formaldehyde_group)]
fig = draw_dist_for_group(formaldehyde_group, formaldehyde_vars)
fig.show()

In [ ]:
vvaerosolindex_group = "Uvaerosolindex"
vvaerosolindex_vars = train.columns[train.columns.str.startswith(vvaerosolindex_group)]
fig = draw_dist_for_group(vvaerosolindex_group, vvaerosolindex_vars, height=540)
fig.show()

In [ ]:
ozone_group = "Ozone"
ozone_vars = train.columns[train.columns.str.startswith(ozone_group)]
fig = draw_dist_for_group(ozone_group, ozone_vars)
fig.show()

In [ ]:
cloud_group = "Cloud"
cloud_vars = train.columns[train.columns.str.startswith(cloud_group)]
fig = draw_dist_for_group(cloud_group, cloud_vars, height=740)
fig.show()

<p style="
    font-size: 20px;
    font-family: 'JetBrains Mono';
    color: #4A4B52;
    border-bottom: 2px solid #E04C5F;
">
    <b>Measurement Distributions</b> 📜
</p>

<p style="
    font-size: 16px;
    font-family: 'JetBrains Mono';
    text-align: justify;
    text-justify: inter-word;
">
    <b>All available samples were used to create the above distributions, so the results should be reliable. Let's say something about each of the features group:</b>
</p>

<ul style="
    font-size: 16px;
    font-family: 'JetBrains Mono';
    text-align: justify;
    text-justify: inter-word;
">
    <li><code>SulphurDioxide-Variables</code> - In all cases, both training and test distributions follow each other. We can assume that the values in the training and test dataset derive from the same distributions. The <code>Sulphurdioxide_Sensor_Azimuth_Angle</code> is quite interesting. This feature denotes the azimuth angle of the satellite at the ground pixel location (WGS84); angle measured East-of-North. It can take values from $-180$ degrees up to $180$ degrees. Here we can see several peaks, so these values usually are constant. As it will turn out, all <code>Sensor_Azimuth_Angle-Variables</code> have similar patterns regardless group. These features usually have several constant values that fluctuate slightly.</li>
    <li><code>CarbonMonoxide-Variables</code> - Similarly as earlier, we have quite good overlaying here. In all cases, both training and test features from that group derive probably from the same distributions.</li>
    <li><code>NitrogenDioxide-Variables</code> - The situation is quite different here. As long as a majority of features' values overlaying, the four features have quite different characters. These are <code>Nitrogendioxide_No2_Column_Number_Density</code>, <code>Nitrogendioxide_Stratospheric_No2_Column_Number_Density</code>, <code>Nitrogendioxide_No2_Slant_Column_Number_Density</code> and <code>Nitrogendioxide_Absorbing_Aerosol_Index</code>. This is the most visible in the <code>Nitrogendioxide_Stratospheric_No2_Column_Number_Density</code>, where distributions rather follow the normal one, but values in the training dataset have completely different mean value than in test one. As we can read, this feature means a stratospheric vertical column of NO2. I checked each year in the training dataset, and still, there is a similar difference regarding the test dataset, i.e. in the test dataset, the mean value is around $20$% higher. What caused such a difference in $2022$?</li>
    <li><code>Formaldehyde-Variables</code> - There is nothing suspect. Almost perfect overlaying.</li>
    <li><code>UVAerosolIndex-Variables</code> - Here one sub-feature recorded an increase in $2022$ (test dataset). The <code>Uvaerosolindex_Absorbing_Aerosol_Index</code>, where the mean value increased from $-1.34$ to $-0.74$. This feature is a measure of the prevalence of aerosols in the atmosphere.</li>
    <li><code>Ozone-Variables</code> - The completely different character of the <code>Ozone_O3_Column_Number_Density</code> feature. It's the total atmospheric column of O3 between the surface and the top of the atmosphere. Moreover, the <code>Ozone_O3_Effective_Temperature</code> recorded a slight increase. This feature is an ozone cross section effective temperature.</li>
    <li><code>Cloud-Variables</code> - Almost perfect overlaying in all sub-features.</li>
</ul>

<p style="
    font-size: 16px;
    font-family: 'JetBrains Mono';
    text-align: justify;
    text-justify: inter-word;
">
    <b>Let's summarize the differences we spot. The following features recorded clearly visible increases in their values:</b>
    <ul style="
        font-size: 16px;
        font-family: 'JetBrains Mono';
    "> 
        <li><code>Nitrogendioxide_No2_Column_Number_Density</code> - total vertical column of NO2 (ratio of the slant column density of NO2 and the total air mass factor),</li>
        <li><code>Nitrogendioxide_Stratospheric_No2_Column_Number_Density</code> - stratospheric vertical column of NO2,</li> <li><code>Nitrogendioxide_No2_Slant_Column_Number_Density</code> - NO2 slant column density,</li>
        <li><code>Nitrogendioxide_Absorbing_Aerosol_Index</code> - aerosol index,</li>
        <li><code>Uvaerosolindex_Absorbing_Aerosol_Index</code> - a measure of the prevalence of aerosols in the atmosphere,</li> <li><code>Ozone_O3_Effective_Temperature</code> - ozone cross section effective temperature.</li> 
    </ul>
</p>

<p style="
    font-size: 16px;
    font-family: 'JetBrains Mono';
    text-align: justify;
    text-justify: inter-word;
">
    Moreover, the character of <code>Ozone_O3_Column_Number_Density</code> (total atmospheric column of O3 between the surface and the top of atmosphere) has completely changed. As for the remaining, we can say that these are on the same level.<br><br>
    <b>Insight</b> - I asked ChatGPT about what can cause these changes. There is an answer: <i>Overall, the observed increase in these atmospheric variables in $2022$ could be a result of several interconnected factors, including industrial activities, transportation, biomass burning, and other anthropogenic emissions, as well as natural processes like volcanic eruptions and climate variability.</i><br><br>
    One thing was interesting - volcanic eruption. And you know what? <b>There actually was a Mount Nyiragongo eruption in May $2021$ in the Democratic Republic of the Congo. See this: <a href="https://en.wikipedia.org/wiki/2021_Mount_Nyiragongo_eruption">2021 Mount Nyiragongo eruption</a>. I've marked it on the map below.</b>
</p>

In [ ]:
mount_nyiragongo_coord = (-1.519167, 29.254167)

fig = px.scatter_mapbox(
    geo_mean_emission,
    lat="Latitude",
    lon="Longitude",
    color_discrete_sequence=["#3C5880"],
    zoom=7,
    width=840,
    height=840,
    title="Data Collection Locations & Mount Nyiragongo Eruption<br>"
    "<span style='font-size: 75%; font-weight: bold;'>"
    "Mount Nyiragongo began erupting on 22 May 2021</span>",
)
fig.add_scattermapbox(
    lat=[mount_nyiragongo_coord[0]],
    lon=[mount_nyiragongo_coord[1]],
    name="Mount Nyiragongo Eruption",
    marker=dict(color="#E04C5F", size=40, symbol="circle", opacity=0.75),
)
fig.update_layout(
    mapbox_style="open-street-map",
    margin=dict(r=0, t=90, l=0, b=0),
    font_color=FONT_COLOR,
    title_font_size=18,
    legend=dict(yanchor="bottom", xanchor="right", y=1, x=1, orientation="h"),
    plot_bgcolor=BACKGROUND_COLOR,
    paper_bgcolor=BACKGROUND_COLOR,
)

fig.show()


<p style="
    font-size: 20px;
    font-family: 'JetBrains Mono';
    color: #4A4B52;
    border-bottom: 2px solid #E04C5F;
">
    <b>Peculiar Finding</b> 📜
</p>

<p style="
    font-size: 16px;
    font-family: 'JetBrains Mono';
    text-align: justify;
    text-justify: inter-word;
">
    As you can see, this volcano is almost in the centre. That would explain the increase in the variables we mentioned before. Nevertheless, as I could know this peculiar finding is not directly related to CO2 emission, so it rather cannot help us.
</p>

## <b> <span style="font-family: 'JetBrains Mono'; color: #4A4B52">2.2</span> <span style='color: #E04C5F'>|</span> <span style="font-family: 'JetBrains Mono'; color: #4A4B52">Time-Series</span></b><a class="anchor" id="time_series"></a> [↑](#top)

<p style="
    font-size: 16px;
    font-family: 'JetBrains Mono';
    text-align: justify;
    text-justify: inter-word;
">
    <b>Let's have a look at how these measurements look over time, and pay special attention to features probably affected by the volcano eruption.</b>
</p>

In [ ]:
def draw_time_series_for_group(
    group,
    vars_to_plot,
    n_cols=3,
    height=640,
    volcano_affected=(
        "Nitrogendioxide_No2_Column_Number_Density",
        "Nitrogendioxide_Stratospheric_No2_Column_Number_Density",
        "Nitrogendioxide_No2_Slant_Column_Number_Density",
        "Nitrogendioxide_Absorbing_Aerosol_Index",
        "Uvaerosolindex_Absorbing_Aerosol_Index",
        "Ozone_O3_Effective_Temperature",
        "Ozone_O3_Column_Number_Density",
    ),
):
    n_rows, axes = get_n_rows_axes(len(vars_to_plot), n_cols)
    fig = make_subplots(
        rows=n_rows,
        cols=n_cols,
        y_title="Value",
        horizontal_spacing=0.1,
        vertical_spacing=0.1,
    )
    fig.update_annotations(font_size=14)
    for frame, color, frame_name in zip((train, test), ("#3C5880", "#E04C5F"), ("Train", "Test")):
        for k, (var, (row, col)) in enumerate(zip(vars_to_plot, axes), start=1):
            var_by_week = frame.groupby("Date")[var].mean().reset_index()
            fig.add_scatter(
                x=var_by_week.Date,
                y=var_by_week[var],
                line=dict(width=0.5, color=color),
                hovertemplate="Date=%{x}<br>Value=%{y}",
                name=frame_name,
                legendgroup=frame_name,
                showlegend=k == 1,
                row=row,
                col=col,
            )
            fig.update_xaxes(
                tickfont_size=7,
                showgrid=False,
                title_text=var,
                titlefont_size=7,
                titlefont_family="Arial Black",
                row=row,
                col=col,
            )
            fig.update_yaxes(tickfont_size=7, showgrid=False, row=row, col=col)
            if var in volcano_affected:
                fig.add_vrect(
                    x0="2021-05-22",
                    x1=test.Date.iloc[-1],
                    line_width=0,
                    fillcolor="rgba(68, 68, 68, 0.05)",
                    row=row,
                    col=col,
                )

    fig.update_layout(
        width=840,
        height=height,
        title=f"{group} Features - Time Series",
        font_color=FONT_COLOR,
        title_font_size=18,
        plot_bgcolor=BACKGROUND_COLOR,
        paper_bgcolor=BACKGROUND_COLOR,
        legend=dict(yanchor="bottom", xanchor="right", y=1, x=1, orientation="h", title=""),
    )
    return fig


fig = draw_time_series_for_group(sulphurdioxide_group, sulphurdioxide_vars)
fig.show()


In [ ]:
fig = draw_time_series_for_group(carbonmonoxide_group, carbonmonoxide_vars)
fig.show()


In [ ]:
fig = draw_time_series_for_group(nitrogendioxide_group, nitrogendioxide_vars, height=740)
fig.show()


In [ ]:
fig = draw_time_series_for_group(formaldehyde_group, formaldehyde_vars)
fig.show()


In [ ]:
fig = draw_time_series_for_group(vvaerosolindex_group, vvaerosolindex_vars, height=540)
fig.show()


In [ ]:
fig = draw_time_series_for_group(ozone_group, ozone_vars)
fig.show()


In [ ]:
fig = draw_time_series_for_group(cloud_group, cloud_vars, height=740)
fig.show()


<p style="
    font-size: 20px;
    font-family: 'JetBrains Mono';
    color: #4A4B52;
    border-bottom: 2px solid #E04C5F;
">
    <b>Time Series</b> 📜
</p>

<p style="
    font-size: 16px;
    font-family: 'JetBrains Mono';
    text-align: justify;
    text-justify: inter-word;
">
    <b>About Volcano Affected Features:</b> The growth of features' values is clearly visible for <code>Nitrogendioxide_Absorbing_Aerosol_Index</code> and <code>Uvaerosolindex_Absorbing_Aerosol_Index</code>. Moreover, we can it also notice in <code>Nitrogendioxide_No2_Column_Number_Density</code>, <code>Nitrogendioxide_Stratospheric_No2_Column_Number_Density</code>, and <code>Nitrogendioxide_No2_Slant_Column_Number_Density</code>. Nevertheless, this cannot be said in the case of Ozone-like variables, i.e. <code>Ozone_O3_Effective_Temperature</code> and <code>Ozone_O3_Column_Number_Density</code>.<br><br>
    <b>About Other Features:</b> In the remaining features, patterns seem to be the same for the training and test dataset. Moreover, pay attention to the <code>Solar_Azimuth_Angle-Variables</code>. These are sine functions with yearly frequency and almost the same amplitude. A similar situation is for <code>Solar_Zenith_Angle-Variables</code>, but here the frequency is half-yearly.<br><br>
    <b>I played a little with these features from satellite, but it seems that these don't have predictive power. Probably those are real, but emissions are generated artificially. Moreover, there is no correlation between the target and measurements. Therefore, let's leave them now.</b>
</p>

# <b> <span style="font-family: 'JetBrains Mono'; color: #4A4B52">3</span> <span style='color: #E04C5F'>|</span> <span style="font-family: 'JetBrains Mono'; color: #4A4B52">Imputation &amp; Feature Engineering</span></b><a class="anchor" id="imputation_and_feature_engineering"></a> [↑](#top)


<p style="
    font-size: 20px;
    font-family: 'JetBrains Mono';
    color: #4A4B52;
    border-bottom: 2px solid #E04C5F;
">
    <b>About Section</b> 💡
</p>

<p style="
    font-size: 16px;
    font-family: 'JetBrains Mono';
    text-align: justify;
    text-justify: inter-word;
">
    In this section, we will tackle missing values in datasets, and try to provide some additive features related to geospatial locations.
</p>

## <b> <span style="font-family: 'JetBrains Mono'; color: #4A4B52">3.1</span> <span style='color: #E04C5F'>|</span> <span style="font-family: 'JetBrains Mono'; color: #4A4B52">Missing Values</span></b><a class="anchor" id="missing_values"></a> [↑](#top)

<p style="
    font-size: 16px;
    font-family: 'JetBrains Mono';
    text-align: justify;
    text-justify: inter-word;
">
    Let's get started with missing values related to measurements. I know that I just said that these are probably not useful here, but who knows? Maybe there are some hiding patterns. So, let's try to impute these missing ones appropriately. <b>The most logical way (for me) to do that is to utilise each location by week first. If any missing values remain, then we use location and month.</b> And then? Possibly we should use values from the closest locations, but this may take much time to work on a solution, so we leave them then.<br><br>
    To solve that, here we have the <code>CustomImputer</code> class, which takes the dataset and calculates median values by keys. It just uses the <code>groupby</code> API to do that. Subsequently, these values are used to fill missing ones. Unfortunately, this solution is highly time-consuming.
</p>

In [ ]:
class CustomImputer(BaseEstimator, TransformerMixin):
    def __init__(self, keys):
        self.keys = keys

    def fit(self, X, y=None):
        self.median_values_ = X.groupby(self.keys).median(numeric_only=True)
        return self

    def transform(self, X, y=None):
        return X.swifter.groupby(self.keys, group_keys=False).apply(
            lambda g: self._fillna_with_median(g)  # type: ignore
        )

    def _fillna_with_median(self, group):
        group_key = tuple(group[self.keys].values[0])
        group_vals = self.median_values_.loc[group_key]
        return group.fillna(group_vals)


In [ ]:
# Alternative implementation - more intuitive but slower.

# class CustomImputer(BaseEstimator, TransformerMixin):
#     def __init__(self, keys):
#         self.keys = keys

#     def fit(self, X, y=None):
#         self.median_values_ = X.groupby(self.keys).median(numeric_only=True)
#         return self

#     def transform(self, X, y=None):
#         Xc = X.copy()
#         for group_key, group_values in self.median_values_.iterrows():
#             group_mask = np.all(Xc[self.keys].values == group_key, axis=1)
#             Xc[group_mask] = Xc[group_mask].fillna(group_values)
#         return Xc


In [ ]:
imputer = make_pipeline(
    CustomImputer(["Coordinates", "Week_No"]),
    CustomImputer(["Coordinates", "Month_No"]),
)

train_filled = imputer.fit_transform(train)
test_filled = imputer.transform(test)


In [ ]:
train_ratio = len(train_filled[train_filled.isna().any(axis=1)]) / len(
    train[train.isna().any(axis=1)]
)
test_ratio = len(test_filled[test_filled.isna().any(axis=1)]) / len(test[test.isna().any(axis=1)])

print(CLR + "Missing Ratio in Train (After/Before):", RED + f"{train_ratio:.2%}")
print(CLR + "Missing Ratio in Test (After/Before): ", RED + f"{test_ratio:.2%}")


<p style="
    font-size: 20px;
    font-family: 'JetBrains Mono';
    color: #4A4B52;
    border-bottom: 2px solid #E04C5F;
">
    <b>Missing Values</b> 📜
</p>

<p style="
    font-size: 16px;
    font-family: 'JetBrains Mono';
    text-align: justify;
    text-justify: inter-word;
">
    <b>We've reduced missing values ratio to about $2$% of datasets length.</b> It's a decent result, but still we should tackle these several ones. However, since these are not required for now, let's leave that issue as it is.
</p>

## <b> <span style="font-family: 'JetBrains Mono'; color: #4A4B52">3.2</span> <span style='color: #E04C5F'>|</span> <span style="font-family: 'JetBrains Mono'; color: #4A4B52">Fixing Covid Period</span></b><a class="anchor" id="fixing_covid_period"></a> [↑](#top)

<p style="
    font-size: 16px;
    font-family: 'JetBrains Mono';
    text-align: justify;
    text-justify: inter-word;
">
    Now, let's handle the Covid-19 period, where the emission was significantly lower than in other years. We can reduce that gap just by using multipliers computed for $2019$ and $2021$. We will calculate the mean emission for each corresponding week in $2019$ and $2021$, and then divide that values by those during Covid. This way, we obtain multipliers.
</p>

In [ ]:
def fix_covid_period(X, years_to_use=None, eps=1e-6):
    if years_to_use is None:
        years_to_use = ("2019", "2021")
    Xc = X.copy()
    respective_period_emission = np.array(
        [Xc[(Xc.Date > f"{year}-03") & (Xc.Date < f"{year}-08")].Emission for year in years_to_use]
    )
    covid_period_mask = (Xc.Date > "2020-03") & (Xc.Date < "2020-08")
    covid_ratios = respective_period_emission.T.mean(axis=1) / (  # Mean along respective weeks.
        Xc[covid_period_mask].Emission + eps  # Smoothing factor.
    )
    Xc.loc[covid_period_mask, "Emission"] = Xc.loc[covid_period_mask, "Emission"] * covid_ratios
    return Xc


In [ ]:
train_filled_fixed = fix_covid_period(train_filled)


In [ ]:
mean_emission_per_year_and_week = (
    train_filled_fixed.groupby(["Year", "Week_No"]).Emission.mean().reset_index()
)

fig = px.line(
    mean_emission_per_year_and_week,
    x="Week_No",
    y="Emission",
    labels={"Emission": "Mean Emission", "Week_No": "Week Number"},
    color="Year",
    color_discrete_sequence=["#6B6A6A", "#E04C5F", "#6B6A6A"],
    line_dash="Year",
    line_dash_sequence=["dashdot", "solid", "dash"],
    title="Mean CO\u2082 Emission per Week & Year - Including All Locations<br>"
    "<span style='font-size: 75%; font-weight: bold;'>"
    "After fixing the Covid-19 period</span>",
    height=480,
    width=840,
)
fig.update_traces(line_width=1.75, opacity=0.75)
fig.update_layout(
    font_color=FONT_COLOR,
    title_font_size=18,
    plot_bgcolor=BACKGROUND_COLOR,
    paper_bgcolor=BACKGROUND_COLOR,
    xaxis_showgrid=False,
    yaxis_showgrid=False,
    xaxis_range=(-1, 53),
    yaxis_range=(0, 140),
    legend=dict(yanchor="bottom", xanchor="right", y=1, x=1, orientation="h", title=""),
)
fig.show()


<p style="
    font-size: 20px;
    font-family: 'JetBrains Mono';
    color: #4A4B52;
    border-bottom: 2px solid #E04C5F;
">
    <b>Fixing Covid Period</b> 📜
</p>

<p style="
    font-size: 16px;
    font-family: 'JetBrains Mono';
    text-align: justify;
    text-justify: inter-word;
">
    That method seems to work fine, we shift values a little in the year $2020$, but these are still distinct from $2019$ and $2021$.
</p>

## <b> <span style="font-family: 'JetBrains Mono'; color: #4A4B52">3.3</span> <span style='color: #E04C5F'>|</span> <span style="font-family: 'JetBrains Mono'; color: #4A4B52">Geospatial Clustering</span></b><a class="anchor" id="geospatial_clustering"></a> [↑](#top)

<p style="
    font-size: 16px;
    font-family: 'JetBrains Mono';
    text-align: justify;
    text-justify: inter-word;
">
    This sub-section aims to determine several characteristic points on the map and the distances between these points and the nearest data collection locations. So, in other words, we will perform clustering using <code>KMeans</code> to determine cluster centres, but simultaneously use the real distance between these centres and nearest locations.<br><br>
    To realise that idea, we use the <code>CustomClustering</code> class that takes the dataset, groups it by each geospatial location and calculates the mean emission for each of them. Subsequently, it fits <code>KMeans</code> to that matrix and evaluates cluster centres. To provide real distance from the nearest location to the cluster centre, we use the <code>haversine</code> function from the <code>haversine</code> library, which calculates the distance (here, in kilometres) between two points on Earth using their latitude and longitude. All of that takes place in the <code>fit()</code> method. The <code>transform</code> method only merges <code>Cluster</code> and <code>DistanceToClusterCentre</code> features with the given data frame.
</p>

In [ ]:
class CustomClustering(BaseEstimator, TransformerMixin):
    def __init__(self, n_clusters=8, random_state=None, unit="km"):
        self.n_clusters = n_clusters
        self.random_state = random_state
        self.unit = unit

    def fit(self, X, y=None):
        keys = ["Latitude", "Longitude"]
        self._frame = X.assign(Emission=y).groupby(keys, as_index=False).Emission.mean()

        self.clustering_ = make_pipeline(
            StandardScaler(),
            KMeans(self.n_clusters, random_state=self.random_state, n_init="auto"),
        ).fit(self._frame)

        self.scaler_ = self.clustering_["standardscaler"]  # type: ignore
        self.kmeans_ = self.clustering_["kmeans"]  # type: ignore

        # Only geospatial locations (lat, lon).
        self.centers_ = self.scaler_.inverse_transform(self.kmeans_.cluster_centers_)[:, :2]
        clusters = self.clustering_.predict(self._frame)

        self._frame[["Latitude_Cluster", "Longitude_Cluster"]] = self.centers_[clusters]
        self._frame["Cluster"] = clusters
        self._frame["DistanceToClusterCentre"] = self._frame.apply(
            lambda x: haversine(
                (x.Latitude, x.Longitude),
                (x.Latitude_Cluster, x.Longitude_Cluster),
                unit=self.unit,  # type: ignore
            ),
            axis=1,
        )

        return self

    def transform(self, X, y=None):
        return X.merge(
            self._frame[["Latitude", "Longitude", "Cluster", "DistanceToClusterCentre"]],
            on=["Latitude", "Longitude"],
        ).set_index(X.index)


In [ ]:
cl = CustomClustering(n_clusters=9, random_state=42)

train_filled_fixed_extended = cl.fit_transform(train_filled_fixed, train_filled_fixed.Emission)
test_filled_extended = cl.transform(test_filled)

train_filled_fixed_extended[
    ["Latitude", "Longitude", "Cluster", "DistanceToClusterCentre"]
].head().style.set_table_styles(DF_STYLE)


In [ ]:
fig = px.scatter_mapbox(
    geo_mean_emission,
    lat="Latitude",
    lon="Longitude",
    color="Emission",
    size="Emission",
    color_continuous_scale=px.colors.sequential.Cividis,
    size_max=30,
    zoom=7,
    width=840,
    height=840,
    title="Distinct Locations of Data Collection in Rwanda",
)
fig.add_scattermapbox(
    lat=zero_emission.Latitude,
    lon=zero_emission.Longitude,
    name="Zero-Emission",
    marker=dict(color="#E04C5F", size=15, symbol="circle", opacity=0.75),
)
fig.add_scattermapbox(
    lat=cl.centers_[:, 0],
    lon=cl.centers_[:, 1],
    name="KMeans-Cluster",
    marker=dict(color="#3E3F4C", size=30, symbol="circle", opacity=0.75),
)
fig.update_layout(
    mapbox_style="open-street-map",
    margin=dict(r=0, t=70, l=0, b=0),
    font_color=FONT_COLOR,
    title_font_size=18,
    coloraxis_colorbar=dict(
        title="Mean Emission",
        orientation="h",
        title_side="top",
        yanchor="bottom",
        xanchor="center",
        y=-0.13,
        x=0.5,
    ),
    legend=dict(yanchor="bottom", xanchor="right", y=1, x=1, orientation="h"),
    plot_bgcolor=BACKGROUND_COLOR,
    paper_bgcolor=BACKGROUND_COLOR,
)
fig.show()


<p style="
    font-size: 20px;
    font-family: 'JetBrains Mono';
    color: #4A4B52;
    border-bottom: 2px solid #E04C5F;
">
    <b>Geospatial Clustering</b> 📜
</p>

<p style="
    font-size: 16px;
    font-family: 'JetBrains Mono';
    text-align: justify;
    text-justify: inter-word;
">
    The <code>KMeans</code> algorithm determined clusters in the centre of Rwanda, between the two highest emission locations, and on the district peripheries.<br><br>
    <b>Whether such features can positively influence the model? Hard to say, probably, these can introduce a little bit of information, but that's all I can expect.</b>
</p>

# <b> <span style="font-family: 'JetBrains Mono'; color: #4A4B52">4</span> <span style='color: #E04C5F'>|</span> <span style="font-family: 'JetBrains Mono'; color: #4A4B52">Machine Learning Model</span></b><a class="anchor" id="machine_learning_model"></a> [↑](#top)

<p style="
    font-size: 20px;
    font-family: 'JetBrains Mono';
    color: #4A4B52;
    border-bottom: 2px solid #E04C5F;
">
    <b>About Section</b> 💡
</p>

<p style="
    font-size: 16px;
    font-family: 'JetBrains Mono';
    text-align: justify;
    text-justify: inter-word;
">
    In this section, we will build several machine-learning models to forecast emissions in $2022$:
    <ul style="
        font-size: 16px;
        font-family: 'JetBrains Mono';
        text-align: justify;
        text-justify: inter-word;
    ">
        <li>The first model (simple Decision Tree) is the most basic and straightforward since it uses only three features and data with a fixed Covid period.</li>
        <li>The second one should be a little bit more intelligent (we will use Random Forest). We introduce there features we derived in the previous section and perform postprocessing.</li>
        <li>The third model is a completely different approach. It utilises emission decomposition (using singular value decomposition) to forecast only five main components. Each of the $497$ time series can be derived as a linear combination of these five components. We will use Random Forest to forecast those five components.</li>
        <li>The fourth approach is basically the same as the third, but we will use Non-Negative Matrix Factorization (NMF) to decompose emissions.</li>
        <li>The last method supposes decomposition with NMF and utilises of ARIMA models to forecast five main components.</li>
    </ul>
</p>

<p style="
    font-size: 16px;
    font-family: 'JetBrains Mono';
">
    <b>So, let's get started.</b>
</p>

## <b> <span style="font-family: 'JetBrains Mono'; color: #4A4B52">4.1</span> <span style='color: #E04C5F'>|</span> <span style="font-family: 'JetBrains Mono'; color: #4A4B52">Basic Decision Tree</span></b><a class="anchor" id="basic_decision_tree"></a> [↑](#top)

<p style="
    font-size: 16px;
    font-family: 'JetBrains Mono';
    text-align: justify;
    text-justify: inter-word;
">
    We will use <code>DecisionTreeRegressor</code> and only three features from the training dataset, i.e. <code>Latitude</code>, <code>Longitude</code>, and <code>Week_No</code>. Since there are no missing values, we don't need to bother about appropriate imputation, and what is more, this is probably the most basic and straightforward approach. Subsequently, we can develop this approach, for example, by adding measurements from the satellite.
</p>

In [ ]:
X = train_filled_fixed_extended[["Latitude", "Longitude", "Week_No"]]
y = train_filled_fixed_extended.Emission

# Small regularization with `min_samples_leaf`.
model = DecisionTreeRegressor(min_samples_leaf=4).fit(X, y)

submission_basic = pd.DataFrame(
    {
        "ID_LAT_LON_YEAR_WEEK": test_filled_extended.index,
        "Emission": model.predict(test_filled_extended[["Latitude", "Longitude", "Week_No"]]),
    }
).set_index("ID_LAT_LON_YEAR_WEEK")

submission_basic.to_csv("submission_basic.csv")
submission_basic.head().style.set_table_styles(DF_STYLE)


In [ ]:
decision_tree_forecast = submission_basic.assign(
    Date=test_filled_extended.Date, Coordinates=test_filled_extended.Coordinates
)

fig = go.Figure()

for df, color in zip(
    (train_filled_fixed_extended, decision_tree_forecast),
    ("rgba(60, 88, 128, 0.3)", "rgba(224, 76, 95, 0.3)"),
):
    for coordinates in df.Coordinates.unique():
        frame = df[df.Coordinates == coordinates]
        fig.add_scatter(
            name="",
            x=frame.Date,
            y=frame.Emission,
            showlegend=False,
            line=dict(width=0.5, color=color),
            text=frame.Coordinates,
            hovertemplate="Coordinates=%{text}<br>Date=%{x}<br>Emission=%{y}",
        )
fig.add_vline(x="2022-01-01", line_width=2, line_dash="dash", line_color="#2D425E")
fig.add_annotation(
    x="2022-07-01",
    y=np.log(7e3) / np.log(10),
    text="<b>Forecast</b>",
    showarrow=False,
    font_size=12,
)
fig.add_annotation(
    x="2021-06-01",
    y=np.log(7e3) / np.log(10),
    text="<b>Training Data</b>",
    showarrow=False,
    font_size=12,
)
fig.update_xaxes(title="Date", showgrid=False, range=("2018-12-01", "2022-12-31"))
fig.update_yaxes(title="Emission", showgrid=False, type="log")
fig.update_layout(
    width=840,
    height=540,
    title="Simple Forecast with Decision Tree<br>"
    "<span style='font-size: 75%; font-weight: bold;'>"
    "Each line is associated with a certain geospatial location</span>",
    font_color=FONT_COLOR,
    title_font_size=18,
    plot_bgcolor=BACKGROUND_COLOR,
    paper_bgcolor=BACKGROUND_COLOR,
    legend=dict(yanchor="bottom", xanchor="right", y=1, x=1, orientation="h", title=""),
)
fig.show()


<p style="
    font-size: 20px;
    font-family: 'JetBrains Mono';
    color: #4A4B52;
    border-bottom: 2px solid #E04C5F;
">
    <b>Forecast - Observations</b> 📜
</p>

<p style="
    font-size: 16px;
    font-family: 'JetBrains Mono';
    text-align: justify;
    text-justify: inter-word;
">
    So, it looks like that. Could I tell more? Considering we've used only geographical location and the week of the observation, it's fine.
</p>

## <b> <span style="font-family: 'JetBrains Mono'; color: #4A4B52">4.2</span> <span style='color: #E04C5F'>|</span> <span style="font-family: 'JetBrains Mono'; color: #4A4B52">Better Approach with Random Forest</span></b><a class="anchor" id="better_approach_with_random_forest"></a> [↑](#top)

<p style="
    font-size: 16px;
    font-family: 'JetBrains Mono';
    text-align: justify;
    text-justify: inter-word;
">
    In this subsection, we will use features we prepared in the previous section and build a simple Random Forest. <b>Moreover, I provide a better CV scheme, i.e. split between public and private LB.</b> What I mean? Well, we know that public LB is evaluated on the first $20$ weeks of $2022$ (see this post: <a href="https://www.kaggle.com/competitions/playground-series-s3e20/discussion/428405"><b>Be cautious- Episode 19 repeats here!</b></a>), and private LB will be evaluated on the rest. So it's good to validate our model on these periods. See below:
</p>

In [ ]:
split_groups = train_filled_fixed_extended.Week_No.between(0, 20).replace(  # type: ignore
    {True: "PublicLB_", False: "PrivateLB_"}
) + train_filled_fixed_extended.Date.dt.year.astype(str)

print(f"{CLR}Groups to Validate:", RED)
print(*split_groups.unique(), sep=", ")


In [ ]:
features = ["Latitude", "Longitude", "Week_No", "Year", "Cluster", "DistanceToClusterCentre"]
train_filled_fixed_extended[features].head().style.set_table_styles(DF_STYLE)


In [ ]:
X = train_filled_fixed_extended[features]
y = train_filled_fixed_extended.Emission

logo = LeaveOneGroupOut()
model = RandomForestRegressor(min_samples_leaf=8, random_state=42)
residuals = np.zeros_like(y)
scores = {}

for train_ids, valid_ids in logo.split(X, y, groups=split_groups):
    X_train, y_train = X.iloc[train_ids], y.iloc[train_ids]
    X_valid, y_valid = X.iloc[valid_ids], y.iloc[valid_ids]

    model.fit(X_train, y_train)
    y_pred = model.predict(X_valid)

    valid_group = split_groups.iloc[valid_ids[0]]
    scores[valid_group] = mean_squared_error(y_valid, y_pred, squared=False)
    residuals[valid_ids] = y_valid - y_pred


In [ ]:
scores_df = pd.DataFrame({"Group": scores.keys(), "RMSE": scores.values()})
public = scores_df[scores_df.Group.str.startswith("Public")].RMSE
private = scores_df[scores_df.Group.str.startswith("Private")].RMSE

print(
    CLR + "Public [0, 20] - Mean Training RMSE:   ",
    f"{RED}{np.mean(public):.2f} \u00B1 {np.std(public):.2f}",
)
print(
    CLR + "Private [21, 52] - Mean Training RMSE: ",
    f"{RED}{np.mean(private):.2f} \u00B1 {np.std(private):.2f}",
)


<p style="
    font-size: 20px;
    font-family: 'JetBrains Mono';
    color: #4A4B52;
    border-bottom: 2px solid #E04C5F;
">
    <b>CV Scheme</b> 📜
</p>

<p style="
    font-size: 16px;
    font-family: 'JetBrains Mono';
    text-align: justify;
    text-justify: inter-word;
">
    As you can see, RMSE is significantly higher for the public LB period than for the private one, even though in the private period we must forecast more values. <b>Additional features bring us some benefits but not so much.</b><br><br>
    <b>Let's see which location introduces the highest error.</b>
</p>

In [ ]:
residual_by_coords = pd.DataFrame(
    {
        "Coordinates": train_filled_fixed_extended.Coordinates,
        "Residual": residuals,
    }
)

highest_rmse = (
    residual_by_coords.groupby("Coordinates", group_keys=True)
    .Residual.apply(lambda g: np.sqrt(np.sum(np.square(g)) / len(g)))
    .sort_values(ascending=False)
)

print(f"{CLR}{highest_rmse[:5]}")


<p style="
    font-size: 20px;
    font-family: 'JetBrains Mono';
    color: #4A4B52;
    border-bottom: 2px solid #E04C5F;
">
    <b>Highest Errors</b> 📜
</p>

<p style="
    font-size: 16px;
    font-family: 'JetBrains Mono';
    text-align: justify;
    text-justify: inter-word;
">
    Model doesn't handle well with the $(-2.079, 29.321)$ location. As we remember, this location has the second highest mean emission (with seasonal pattern). It turns out that there is better to forecast values from the previous year for this location (see this post: <a href="https://www.kaggle.com/competitions/playground-series-s3e20/discussion/429717"><b>[Trick] Replace the trouble maker in your submission!</b></a>).
</p>

In [ ]:
model.fit(X, y)

submission_better = pd.DataFrame(
    {
        "ID_LAT_LON_YEAR_WEEK": test_filled_extended.index,
        "Emission": model.predict(test_filled_extended[features]),
    }
).set_index("ID_LAT_LON_YEAR_WEEK")

submission_better.loc[test.Coordinates == "(-2.079, 29.321)", "Emission"] = train.loc[
    (train.Year == 2021) & (train.Week_No <= 48) & (train.Coordinates == "(-2.079, 29.321)"),
    "Emission",
].to_numpy()

submission_better = submission_better * 1.07
submission_better.to_csv("submission_better.csv")
submission_better.head().style.set_table_styles(DF_STYLE)


## <b> <span style="font-family: 'JetBrains Mono'; color: #4A4B52">4.3</span> <span style='color: #E04C5F'>|</span> <span style="font-family: 'JetBrains Mono'; color: #4A4B52">Emission Decomposition - SVD</span></b><a class="anchor" id="emission_decomposition_svd"></a> [↑](#top)

<p style="
    font-size: 16px;
    font-family: 'JetBrains Mono';
    text-align: justify;
    text-justify: inter-word;
">
    A more sophisticated approach is to decompose the target emission using <code>TruncatedSVD</code> or <code>PCA</code>. In this way, it turns out that we can explain all $497$ locations using linear combinations of only five main components. See this post: <a href="https://www.kaggle.com/competitions/playground-series-s3e20/discussion/429278"><b>Dimensionality reduction: 5 dimensions are enough</b></a>.<br><br>
    <b>This is a huge dimensionality reduction because all we need to forecast is those five components, and all time-series can be derived with inverse transformation. Let's look at that approach step by step.</b>
</p>

In [ ]:
emission_pivot = train_filled_fixed_extended.pivot(
    index=["Year", "Week_No"],
    columns=["Latitude", "Longitude"],
    values="Emission",
)
emission_pivot.head().style.set_table_styles(DF_STYLE)


In [ ]:
print(CLR + "Emission Pivot Shape:", f"{RED}{emission_pivot.shape}")


<p style="
    font-size: 20px;
    font-family: 'JetBrains Mono';
    color: #4A4B52;
    border-bottom: 2px solid #E04C5F;
">
    <b>SVD Approach - Remarks</b> 📜
</p>

<p style="
    font-size: 16px;
    font-family: 'JetBrains Mono';
    text-align: justify;
    text-justify: inter-word;
">
    Firstly we prepare a pivot table of emissions. Here we have a data frame with $159$ rows (these depict each year and each week in the training data) and $497$ columns (each column is associated with a distinct geospatial location). Values in this data frame are emissions for a given location and date. <b>Now the crucial point is that we treat locations as features and try to reduce dimensionality.</b>
</p>

In [ ]:
n_components = 5  # Number of main components in decomposition.

# We will be training a model with only unique dates.
X_train_decomposition = train_filled_fixed_extended[["Year", "Week_No"]].drop_duplicates()
X_test_decomposition = test_filled_extended[["Year", "Week_No"]].drop_duplicates()

# For plots.
train_dates = extract_date_from_year_and_week(
    X_train_decomposition.Year, X_train_decomposition.Week_No
)
test_dates = extract_date_from_year_and_week(
    X_test_decomposition.Year, X_test_decomposition.Week_No
)


In [ ]:
svd = TruncatedSVD(n_components=n_components).fit(emission_pivot)
y_train_svd = svd.transform(emission_pivot)

for n, ratio in enumerate(svd.explained_variance_ratio_.cumsum()):
    print(
        CLR + "Main Components: " + f"{RED}{n + 1}",
        CLR + "\tExplained Variance Ratio: " + f"{RED}{ratio:.3f}",
    )


<p style="
    font-size: 20px;
    font-family: 'JetBrains Mono';
    color: #4A4B52;
    border-bottom: 2px solid #E04C5F;
">
    <b>Emission Decomposition by Location</b> 📜
</p>

<p style="
    font-size: 16px;
    font-family: 'JetBrains Mono';
    text-align: justify;
    text-justify: inter-word;
">
    There we've just passed the emission pivot to the SVD algorithm to reduce dimensionality. <b>It turns out that only five main components (five different time series) explain $99.9$% of the variance, which is quite incredible. This means that, most probably target was generated artificially.</b><br><br>
    Let's have a look at what these components look like.
</p>

In [ ]:
fig = make_subplots(
    rows=5,
    cols=1,
    y_title="Component Value",
    x_title="Date",
    subplot_titles=("Component 1", "Component 2", "Component 3", "Component 4", "Component 5"),
    shared_xaxes=True,
)
fig.update_annotations(font_size=14)
for row, component in enumerate(range(n_components), start=1):
    fig.add_scatter(
        name=f"Component {component + 1}",
        x=train_dates,
        y=y_train_svd[:, component],
        line_color="#6B6A6A",
        hovertemplate="Date=%{x}<br>Value=%{y}",
        showlegend=False,
        row=row,
        col=1,
    )
    fig.update_xaxes(showgrid=False, range=("2018-12-01", "2022-01-01"))
    fig.update_yaxes(showgrid=False)

fig.update_traces(line_width=1.75, opacity=0.75)
fig.update_layout(
    title="CO\u2082 Emission Decomposition with TruncatedSVD<br>"
    "<span style='font-size: 75%; font-weight: bold;'>"
    "Five main components provide 99.9% of explained variance</span>",
    font_color=FONT_COLOR,
    title_font_size=18,
    title_y=0.94,
    margin_t=130,
    plot_bgcolor=BACKGROUND_COLOR,
    paper_bgcolor=BACKGROUND_COLOR,
    width=840,
    height=740,
    legend=dict(yanchor="bottom", xanchor="right", y=1, x=1, orientation="h", title=""),
)
fig.show()


<p style="
    font-size: 20px;
    font-family: 'JetBrains Mono';
    color: #4A4B52;
    border-bottom: 2px solid #E04C5F;
">
    <b>Emission Decomposition</b> 📜
</p>

<p style="
    font-size: 16px;
    font-family: 'JetBrains Mono';
    text-align: justify;
    text-justify: inter-word;
">
    <ul style="
        font-size: 16px;
        font-family: 'JetBrains Mono';
        text-align: justify;
        text-justify: inter-word;
    ">
        <li><code>Component 1</code> - Related to the location with the highest mean emission, i.e. $(-2.378, 29.222)$, Nyamasheke.</li>
        <li><code>Component 2</code> - Related to the location with the second highest mean emission and seasonal pattern, i.e. $(-2.079, 29.321)$, Karongi.</li>
        <li><code>Component 3</code> - The component which shows a seasonal pattern (peaks in April and October).</li>
        <li><code>Component 4</code> and <code>Component 5</code> - Components with lowest amplitude and without seasonal patterns, slightly shifted from each other.</li>
    </ul>
</p>

<p style="
    font-size: 16px;
    font-family: 'JetBrains Mono';
    text-align: justify;
    text-justify: inter-word;
">
    <b>So, we have five basis time series, which we need to forecast and then derive series for $497$ locations.</b> Here we will use <code>MultiOutputRegressor</code>, which trains a separate model for each time series, using only the year and week.
</p>


In [ ]:
svd_model = MultiOutputRegressor(
    RandomForestRegressor(min_samples_leaf=8, random_state=42),
).fit(X_train_decomposition, y_train_svd)

svd_predictions = svd_model.predict(X_test_decomposition)


In [ ]:
fig = make_subplots(
    rows=5,
    cols=1,
    y_title="Component Value",
    x_title="Date",
    subplot_titles=("Component 1", "Component 2", "Component 3", "Component 4", "Component 5"),
    shared_xaxes=True,
)
fig.update_annotations(font_size=14)
for row, component in enumerate(range(n_components), start=1):
    fig.add_scatter(
        name="Train",
        x=train_dates,
        y=y_train_svd[:, component],
        line_color="rgba(60, 88, 128, 0.75)",
        hovertemplate="Date=%{x}<br>Value=%{y}",
        legendgroup="Train",
        showlegend=row == 1,
        row=row,
        col=1,
    )
    fig.add_scatter(
        name="Test",
        x=test_dates,
        y=svd_predictions[:, component],
        line_color="rgba(224, 76, 95, 0.75)",
        hovertemplate="Date=%{x}<br>Value=%{y}",
        legendgroup="Test",
        showlegend=row == 1,
        row=row,
        col=1,
    )
    fig.add_vline(x="2022-01-01", line_width=2, line_dash="dash", line_color="#2D425E")
    fig.update_xaxes(showgrid=False, range=("2018-12-01", "2022-12-31"))
    fig.update_yaxes(showgrid=False)

fig.update_traces(line_width=1.5)
fig.update_layout(
    title="CO\u2082 Emission Decomposition with TruncatedSVD<br>"
    "<span style='font-size: 75%; font-weight: bold;'>"
    "The red lines depict forecast (using Random Forest) for 2022</span>",
    font_color=FONT_COLOR,
    title_font_size=18,
    title_y=0.94,
    margin_t=130,
    plot_bgcolor=BACKGROUND_COLOR,
    paper_bgcolor=BACKGROUND_COLOR,
    width=840,
    height=740,
    legend=dict(yanchor="bottom", xanchor="right", y=1, x=1, orientation="h", title=""),
)
fig.show()


<p style="
    font-size: 20px;
    font-family: 'JetBrains Mono';
    color: #4A4B52;
    border-bottom: 2px solid #E04C5F;
">
    <b>SVD Forecast</b> 📜
</p>

<p style="
    font-size: 16px;
    font-family: 'JetBrains Mono';
    text-align: justify;
    text-justify: inter-word;
">
    It looks as it looks. The <code>min_samples_leaf=8</code> introduces quite a strong regularization, so only the second and third component looks natural.<br><br>
    <b>Now the only thing we need to make is to derive the original time series for the forecast using inverse transformation.</b>
</p>


In [ ]:
indices = test_filled_extended[["Year", "Week_No"]].drop_duplicates().to_numpy().T

test_svd_pivot = pd.DataFrame(
    svd.inverse_transform(svd_predictions),
    columns=emission_pivot.columns,
    index=pd.MultiIndex.from_arrays(indices, names=("Year", "Week_No")),
)
test_svd_pivot.head().style.set_table_styles(DF_STYLE)


In [ ]:
test_svd_forecast = (
    test_svd_pivot.melt(ignore_index=False, value_name="Emission")
    .reset_index()
    .set_index(test_filled_extended.index)
)

test_svd_forecast.head().style.set_table_styles(DF_STYLE)


<p style="
    font-size: 20px;
    font-family: 'JetBrains Mono';
    color: #4A4B52;
    border-bottom: 2px solid #E04C5F;
">
    <b>SVD Approach - Remarks</b> 📜
</p>

<p style="
    font-size: 16px;
    font-family: 'JetBrains Mono';
    text-align: justify;
    text-justify: inter-word;
">
    And done! Now we have the form of forecast for the test dataset as we need. Let's have a look at the derived time series.
</p>


In [ ]:
svd_forecast = test_svd_forecast.assign(
    Date=test_filled_extended.Date, Coordinates=test_filled_extended.Coordinates
)

fig = go.Figure()

for df, color in zip(
    (train_filled_fixed_extended, svd_forecast),
    ("rgba(60, 88, 128, 0.3)", "rgba(224, 76, 95, 0.3)"),
):
    for coordinates in df.Coordinates.unique():
        frame = df[df.Coordinates == coordinates]
        fig.add_scatter(
            name="",
            x=frame.Date,
            y=frame.Emission,
            showlegend=False,
            line=dict(width=0.5, color=color),
            text=frame.Coordinates,
            hovertemplate="Coordinates=%{text}<br>Date=%{x}<br>Emission=%{y}",
        )
fig.add_vline(x="2022-01-01", line_width=2, line_dash="dash", line_color="#2D425E")
fig.add_annotation(
    x="2022-07-01",
    y=np.log(7e3) / np.log(10),
    text="<b>Forecast</b>",
    showarrow=False,
    font_size=12,
)
fig.add_annotation(
    x="2021-06-01",
    y=np.log(7e3) / np.log(10),
    text="<b>Training Data</b>",
    showarrow=False,
    font_size=12,
)
fig.update_xaxes(title="Date", showgrid=False, range=("2018-12-01", "2022-12-31"))
fig.update_yaxes(title="Emission", showgrid=False, type="log")
fig.update_layout(
    width=840,
    height=540,
    title="Forecast with SVD (99.9% variance explained) and Random Forest<br>"
    "<span style='font-size: 75%; font-weight: bold;'>"
    "Each line is associated with a certain geospatial location</span>",
    font_color=FONT_COLOR,
    title_font_size=18,
    plot_bgcolor=BACKGROUND_COLOR,
    paper_bgcolor=BACKGROUND_COLOR,
    legend=dict(yanchor="bottom", xanchor="right", y=1, x=1, orientation="h", title=""),
)
fig.show()


<p style="
    font-size: 20px;
    font-family: 'JetBrains Mono';
    color: #4A4B52;
    border-bottom: 2px solid #E04C5F;
">
    <b>Derived Time Series</b> 📜
</p>

<p style="
    font-size: 16px;
    font-family: 'JetBrains Mono';
">
    At first sight, it looks fine. Now we prepare submission and postprocessing.
</p>


In [ ]:
submission_svd = pd.DataFrame(
    {
        "ID_LAT_LON_YEAR_WEEK": test_svd_forecast.index,
        "Emission": test_svd_forecast.Emission,
    }
).set_index("ID_LAT_LON_YEAR_WEEK")

submission_svd.loc[test.Coordinates == "(-2.079, 29.321)", "Emission"] = train.loc[
    (train.Year == 2021) & (train.Week_No <= 48) & (train.Coordinates == "(-2.079, 29.321)"),
    "Emission",
].to_numpy()

submission_svd = submission_svd * 1.07
submission_svd.to_csv("submission_svd.csv")
submission_svd.head().style.set_table_styles(DF_STYLE)


## <b> <span style="font-family: 'JetBrains Mono'; color: #4A4B52">4.4</span> <span style='color: #E04C5F'>|</span> <span style="font-family: 'JetBrains Mono'; color: #4A4B52">Emission Decomposition - NMF</span></b><a class="anchor" id="emission_decomposition_nmf"></a> [↑](#top)

<p style="
    font-size: 16px;
    font-family: 'JetBrains Mono';
    text-align: justify;
    text-justify: inter-word;
">
    Last day there was another great post by <a href="https://www.kaggle.com/ambrosm"><b>AmbrosM</b></a>. See this: <a href="https://www.kaggle.com/competitions/playground-series-s3e20/discussion/430850"><b>NMF explains how the data was generated</b></a>.<br><br>
    There is nothing left but to check this method on our own. The NMF is an acronym for Non-Negative Matrix Factorization. <b>Supposing that emission data for each week and each location is a non-negative signal, this approach seems to be better than ordinary SVD since NMF aims to find non-negative matrices, i.e. matrices with all non-negative elements $(W, H)$ whose product approximates the non-negative matrix $X$ (in this case the emission pivot table).</b> The scheme is practically the same as in the previous subsection. Let's see.
</p>

In [ ]:
nmf = NMF(
    n_components=n_components,
    init="nndsvdar",
    solver="mu",
    beta_loss="kullback-leibler",
    tol=1e-6,
    max_iter=10_000,
    random_state=42,
).fit(emission_pivot)

y_train_nmf = nmf.transform(emission_pivot)
nmf_reconstruction = nmf.inverse_transform(y_train_nmf)

nmf_explained_variance = explained_variance_score(
    emission_pivot, nmf_reconstruction, multioutput="variance_weighted"
)
print(
    CLR + "NMF with 5 components - explained variance ratio:",
    f"{RED}{nmf_explained_variance:.3f}",
)


<p style="
    font-size: 20px;
    font-family: 'JetBrains Mono';
    color: #4A4B52;
    border-bottom: 2px solid #E04C5F;
">
    <b>NMF Approach - Remarks</b> 📜
</p>

<p style="
    font-size: 16px;
    font-family: 'JetBrains Mono';
    text-align: justify;
    text-justify: inter-word;
">
    Here, the most important hyperparameter seems to be the <code>init</code> one. In this case, <i>nndsvdar</i> (Nonnegative Double Singular Value Decomposition initialization with zeros filled with small random values) seems to work the best. Moreover, I noticed that Kullback–Leibler divergence gives a little bit more in this special case than Frobenius Norm.<br><br>
    The <code>NMF</code> doesn't have the <code>explained_variance_ratio_</code> attribute, so we need to calculate it by hand, as we did above. In order to compare this approach with <code>TruncatedSVD</code>, we need to provide a <i>variance_weighted</i> parameter in <code>multioutput</code>. It means that scores of all outputs are averaged, weighted by the variances of each individual output. As we can see explained variance is almost the same as in the <code>TruncatedSVD</code>.<br><br>
    <b>Now we proceed exactly the same way as in the previous subsection.</b>
</p>

In [ ]:
fig = make_subplots(
    rows=5,
    cols=1,
    y_title="Component Value",
    x_title="Date",
    subplot_titles=("Component 1", "Component 2", "Component 3", "Component 4", "Component 5"),
    shared_xaxes=True,
)
fig.update_annotations(font_size=14)
for row, component in enumerate(range(n_components), start=1):
    fig.add_scatter(
        name=f"Component {component + 1}",
        x=train_dates,
        y=y_train_nmf[:, component],
        line_color="#6B6A6A",
        hovertemplate="Date=%{x}<br>Value=%{y}",
        showlegend=False,
        row=row,
        col=1,
    )
    fig.update_xaxes(showgrid=False, range=("2018-12-01", "2022-01-01"))
    fig.update_yaxes(showgrid=False)

fig.update_traces(line_width=1.5, opacity=0.75)
fig.update_layout(
    title="CO\u2082 Emission Decomposition with NMF<br>"
    "<span style='font-size: 75%; font-weight: bold;'>"
    "Five main components provide 99.3% of explained variance</span>",
    font_color=FONT_COLOR,
    title_font_size=18,
    title_y=0.94,
    margin_t=130,
    plot_bgcolor=BACKGROUND_COLOR,
    paper_bgcolor=BACKGROUND_COLOR,
    width=840,
    height=740,
    legend=dict(yanchor="bottom", xanchor="right", y=1, x=1, orientation="h", title=""),
)
fig.show()


In [ ]:
nmf_model = MultiOutputRegressor(
    RandomForestRegressor(min_samples_leaf=8, random_state=42),
).fit(X_train_decomposition, y_train_nmf)

nmf_predictions = nmf_model.predict(X_test_decomposition)


In [ ]:
fig = make_subplots(
    rows=5,
    cols=1,
    y_title="Component Value",
    x_title="Date",
    subplot_titles=("Component 1", "Component 2", "Component 3", "Component 4", "Component 5"),
    shared_xaxes=True,
)
fig.update_annotations(font_size=14)
for row, component in enumerate(range(n_components), start=1):
    fig.add_scatter(
        name="Train",
        x=train_dates,
        y=y_train_nmf[:, component],
        line_color="rgba(60, 88, 128, 0.75)",
        hovertemplate="Date=%{x}<br>Value=%{y}",
        legendgroup="Train",
        showlegend=row == 1,
        row=row,
        col=1,
    )
    fig.add_scatter(
        name="Test",
        x=test_dates,
        y=nmf_predictions[:, component],
        line_color="rgba(224, 76, 95, 0.75)",
        hovertemplate="Date=%{x}<br>Value=%{y}",
        legendgroup="Test",
        showlegend=row == 1,
        row=row,
        col=1,
    )
    fig.add_vline(x="2022-01-01", line_width=2, line_dash="dash", line_color="#2D425E")
    fig.update_xaxes(showgrid=False, range=("2018-12-01", "2022-12-31"))
    fig.update_yaxes(showgrid=False)

fig.update_traces(line_width=1.5)
fig.update_layout(
    title="CO\u2082 Emission Decomposition with NMF<br>"
    "<span style='font-size: 75%; font-weight: bold;'>"
    "The red lines depict forecast (using Random Forest) for 2022</span>",
    font_color=FONT_COLOR,
    title_font_size=18,
    title_y=0.94,
    margin_t=130,
    plot_bgcolor=BACKGROUND_COLOR,
    paper_bgcolor=BACKGROUND_COLOR,
    width=840,
    height=740,
    legend=dict(yanchor="bottom", xanchor="right", y=1, x=1, orientation="h", title=""),
)
fig.show()


<p style="
    font-size: 20px;
    font-family: 'JetBrains Mono';
    color: #4A4B52;
    border-bottom: 2px solid #E04C5F;
">
    <b>NMF Forecast</b> 📜
</p>

<p style="
    font-size: 16px;
    font-family: 'JetBrains Mono';
    text-align: justify;
    text-justify: inter-word;
">
    It looks similar to the SVD approach. Let's prepare the appropriate derived time series and submission. I'm not going to show a graph for each location because it looks very similar to the previous ones.
</p>


In [ ]:
test_nmf_forecast = (
    pd.DataFrame(
        nmf.inverse_transform(nmf_predictions),
        columns=emission_pivot.columns,
        index=pd.MultiIndex.from_arrays(indices, names=("Year", "Week_No")),
    )
    .melt(ignore_index=False, value_name="Emission")
    .reset_index()
    .set_index(test_filled_extended.index)
)

test_nmf_forecast.head().style.set_table_styles(DF_STYLE)


In [ ]:
submission_nmf = pd.DataFrame(
    {
        "ID_LAT_LON_YEAR_WEEK": test_nmf_forecast.index,
        "Emission": test_nmf_forecast.Emission,
    }
).set_index("ID_LAT_LON_YEAR_WEEK")

submission_nmf.loc[test.Coordinates == "(-2.079, 29.321)", "Emission"] = train.loc[
    (train.Year == 2021) & (train.Week_No <= 48) & (train.Coordinates == "(-2.079, 29.321)"),
    "Emission",
].to_numpy()

submission_nmf = submission_nmf * 1.07
submission_nmf.to_csv("submission_nmf.csv")
submission_nmf.head().style.set_table_styles(DF_STYLE)


## <b> <span style="font-family: 'JetBrains Mono'; color: #4A4B52">4.5</span> <span style='color: #E04C5F'>|</span> <span style="font-family: 'JetBrains Mono'; color: #4A4B52">NMF Approach &amp; ARIMA Models</span></b><a class="anchor" id="nmf_approach_and_arima_models"></a> [↑](#top)

<p style="
    font-size: 16px;
    font-family: 'JetBrains Mono';
    text-align: justify;
    text-justify: inter-word;
">
    In this subsection, we will focus on blending the NMF approach with ARIMA models to forecast five main components.<br><br>
    ARIMA (AutoRegressive Integrated Moving Average) is another forecasting method, but completely different from tree-based ones. <b>ARIMA model account for the combination of autoregressive models (predictions based on linear combinations of last observations) and moving average models (past forecast errors).</b> We write it as the ARIMA$(p, d, q)$ model, where $p$ denotes the order of the autoregressive part, $d$ is the degree of first differencing involved, and $q$ is the order of the moving average part. Generally, ARIMA is a non-seasonal model. Nevertheless, seasonality can be included in these models. We call it ARIMA$(p, d, q)(P, D, Q)_m$, where the second part $(P, D, Q)_m$ is liable for the seasonality and $m$ denotes the number of observations per season.<br><br>
    Before we take to the ARIMA method, we need to say something about stationarity in time series. See below:
</p>

In [ ]:
np.random.seed(42)

# Generate gaussian noise with the same dates as training dataset.
periods = 159
noise = np.random.randn(periods)
trend_noise = noise + np.exp(2 * np.arange(periods) / periods)
dates = pd.date_range("2019-01-01", periods=periods, freq="7D")


In [ ]:
fig = make_subplots(
    rows=2,
    cols=1,
    y_title="Value",
    x_title="Date",
    subplot_titles=(
        "Gaussian Noise - Stationary Signal",
        "Trended Gaussian Noise - Non-Stationary Signal",
    ),
    shared_xaxes=True,
)
fig.update_annotations(font_size=14)

for row, series in enumerate((noise, trend_noise), start=1):
    fig.add_scatter(
        name="Train",
        x=dates,
        y=series,
        line_color="#6B6A6A",
        hovertemplate="Date=%{x}<br>Value=%{y}",
        legendgroup="Train",
        showlegend=False,
        row=row,
        col=1,
    )
    fig.update_xaxes(showgrid=False, range=("2018-12-01", "2022-01-01"))
    fig.update_yaxes(showgrid=False)

fig.update_traces(line_width=1.75, opacity=0.75)
fig.update_layout(
    title="Examples of Stationary / Non-Stationary Time Series",
    font_color=FONT_COLOR,
    title_font_size=18,
    title_y=0.92,
    margin_t=130,
    plot_bgcolor=BACKGROUND_COLOR,
    paper_bgcolor=BACKGROUND_COLOR,
    legend=dict(yanchor="bottom", xanchor="right", y=1, x=1, orientation="h", title=""),
    width=840,
    height=540,
)
fig.show()


<p style="
    font-size: 20px;
    font-family: 'JetBrains Mono';
    color: #4A4B52;
    border-bottom: 2px solid #E04C5F;
">
    <b>Stationary Time Series - Remarks</b> 💡
</p>

<p style="
    font-size: 16px;
    font-family: 'JetBrains Mono';
    text-align: justify;
    text-justify: inter-word;
">
    <b>Generally, in stationary time series, its properties do not depend on the time at which the data are observed (but it isn't so easy either).</b> Therefore, time series data with trends or seasonality are not stationary because these components affect the values we observe. Nevertheless, this no so obvious, and some cases can be confusing. For example, cyclic time series (with no seasonal pattern and trend) may be stationary too. Why? Because cycles usually haven't got fixed periods, so, we cannot be sure where peaks and depressions will be in the future. This causes the cyclic time series to be unpredictable.<br><br>
    <b>In other words, the stationary time series is unpredictable over the long-time. The first time series (Gaussian noise) you see above is stationary indeed. Statistical properties of this signal do not depend on time, and there is no difference when do you observe it. However, the second one is non-stationary due to the upward trend.</b><br><br>
    <b>The second time series is non-stationary but was generated using a stationary component. How can we make non-stationary series a stationary one? That's the task of differencing - computation of differences between consecutive observations, i.e. $y_t' = y_t - y_{t-1}$.</b> Sometimes differenced time series still will not be stationary, and we need to difference it the second time, i.e. $y_t'' = y_t' - y_{t-1}'$. We call it a second-order differencing. Moreover, there is something we call seasonal differencing, i.e. differences between this and the previous observation from the same season.<br><br>
    <b>How to determine whether differencing is required in the ARIMA model? We can use the KPSS test or the Augmented Dickey-Fuller test.</b> Let's take generated time series and check them with the second method.
</p>

In [ ]:
def adf_results(series, name):
    result = adfuller(series)
    print(CLR + f"Augmented Dickey-Fuller Test - {name}:")
    print(CLR + "=" * 72)
    print(CLR + "● critical value:".ljust(52), f"{RED}{result[0]:.2f}")
    print(CLR + "● p-value:".ljust(52), f"{RED}{result[1]:.5f}")
    print(CLR + "● number of lags used:".ljust(52), f"{RED}{result[2]}")
    print(CLR + "● number of observations:".ljust(52), f"{RED}{result[3]}")
    print(
        CLR + "● t-values at 1%, 5% and 10% confidence intervals:".ljust(52),
        f"{RED}{np.array(list(result[4].values())).round(2)}",
    )
    print(CLR + "=" * 72, "\n")


In [ ]:
adf_results(noise, "Gaussian Noise")
adf_results(trend_noise, "Trended Gaussian Noise")


<p style="
    font-size: 20px;
    font-family: 'JetBrains Mono';
    color: #4A4B52;
    border-bottom: 2px solid #E04C5F;
">
    <b>Augmented Dickey-Fuller Test</b> 📜
</p>

<p style="
    font-size: 16px;
    font-family: 'JetBrains Mono';
    text-align: justify;
    text-justify: inter-word;
">
    Well, so we got some values, but what are these values? <b>In the Augmented Dickey-Fuller test, the null hypothesis is that the data are non-stationary, and we're looking for evidence that the null hypothesis is false.</b> In consequence, small p-values (usually less than $0.05$) suggest that differencing is not required, which means the signal is stationary, and we should reject the null hypothesis with a $95$% level of confidence.<br><br>
    So, let's take a look at p-values. For the first signal, it's virtually zero, which suggests the signal is stationary (and that's actually true). For the second one p-value is almost equal to one, which suggests, that the differencing is required and the passed signal is not-stationary (that's also true since we add a trend to that series).<br><br>
    It may all seem simple to you now, but stationarity is not so easy at all. Nevertheless, do not take a deep dive into this for now, since it's not a notebook about stationarity details. Let's check the five main components of <code>NMF</code>.
</p>

In [ ]:
for n in range(n_components):
    adf_results(y_train_nmf[:, n], f"NMF - Component {n + 1}")


<p style="
    font-size: 20px;
    font-family: 'JetBrains Mono';
    color: #4A4B52;
    border-bottom: 2px solid #E04C5F;
">
    <b>Augmented Dickey-Fuller Test Results</b> 📜
</p>

<p style="
    font-size: 16px;
    font-family: 'JetBrains Mono';
    text-align: justify;
    text-justify: inter-word;
">
    Well, the third component is surprising. The p-value is zero there, even though there we have a seasonal pattern. The point is that the ADF test is tailored for detecting non-stationarity in the form of a unit root in the process. Nevertheless, it's not adapted for detecting other forms of non-stationarity. Why are we talking about stationarity? Well, in the ARIMA model, the <b>I($d$)</b> part, accounts for the differencing procedure, which makes stationary time series out of the non-stationary one.<br><br>
    <b>Let's leave the stationarity problem and get to the ARIMA modelling.</b><br><br>
    The <b>AR($p$)</b> is an abbreviation of an autoregressive model of order $p$, and can be written as follows:
    \[y_t = c + \phi_1 y_{t-1} + \phi_2 y_{t-2} + \ldots + \phi_p y_{t-p} + \varepsilon_t,\]
    where $\varepsilon_t$ is a white noise.<br><br>
    On the other hand, the  <b>MA($q$)</b> denotes a moving average model of order $q$, and we write it as:
    \[y_t = c + \varepsilon_t + \theta_1\varepsilon_{t-1} + \theta_2\varepsilon_{t-2} + \ldots + \theta_q\varepsilon_{t-q},\]
    where $\varepsilon_t$ is a white noise.<br><br>
    <b>Several combinations of seasonal ARIMA are shown below:</b>
</p>

In [ ]:
p = d = q = range(0, 2)
pdq = list(product(p, d, q))
s_pdq = list((*combination, 53) for combination in pdq)

np.random.seed(42)

for i_pdq, i_spdq in zip(np.random.permutation(pdq), np.random.permutation(s_pdq)):
    print(
        CLR + "ARIMA",
        RED + "({}, {}, {})".format(*i_pdq),
        CLR + "x",
        RED + "({}, {}, {}, {})".format(*i_spdq),
    )


<p style="
    font-size: 20px;
    font-family: 'JetBrains Mono';
    color: #4A4B52;
    border-bottom: 2px solid #E04C5F;
">
    <b>ARIMA Combinations - Remarks</b> 📜
</p>

<p style="
    font-size: 16px;
    font-family: 'JetBrains Mono';
    text-align: justify;
    text-justify: inter-word;
">
    Some of the combinations don't have any sense; for example, ARIMA$(0, 0, 0)$x$(0, 0, 0, 53)$. We can find an appropriate combination by minimalising the Akaike Information Criterion (AIC). Moreover, ARIMA is highly sensitive to dates in the series, so we need to provide that these are valid as below. Otherwise, predictions can be shifted compared to what we need to save.<br><br>
    Subsequently, we train univariate ARIMA models. I found orders and seasonal orders earlier with the AIC approach. Usually, there are no required orders higher than $2$. Here first orders work fine.
</p>

In [ ]:
train_arima_dates = pd.date_range(start="2019-01-01", periods=53 * 3, freq="7D")
first_date_in_test_arima = train_arima_dates[-1] + pd.DateOffset(7)
test_arima_dates = pd.date_range(start=first_date_in_test_arima, periods=49, freq="7D")

arima_forecast = np.zeros((len(test_arima_dates), n_components))


In [ ]:
warnings.simplefilter("ignore")  # Seasonal models require 5 full seasons?

orders = ((1, 1, 1), (1, 1, 1), (0, 1, 1), (1, 1, 0), (1, 1, 0))
s_orders = ((1, 1, 1, 53), (1, 0, 1, 53), (0, 1, 1, 53), (1, 1, 0, 53), (1, 1, 0, 53))

for n, (order, s_order) in enumerate(zip(orders, s_orders)):
    component = pd.Series(y_train_nmf[:, n], index=train_arima_dates.to_period())
    model = ARIMA(component, order=order, seasonal_order=s_order).fit()
    forecast = model.predict(start=test_arima_dates[0], end=test_arima_dates[-1])
    arima_forecast[:, n] = np.where(forecast < 0, 0, forecast)


In [ ]:
fig = make_subplots(
    rows=5,
    cols=1,
    y_title="Component Value",
    x_title="Date",
    subplot_titles=("Component 1", "Component 2", "Component 3", "Component 4", "Component 5"),
    shared_xaxes=True,
)
fig.update_annotations(font_size=14)
for row, component in enumerate(range(n_components), start=1):
    fig.add_scatter(
        name="Train",
        x=train_dates,
        y=y_train_nmf[:, component],
        line_color="rgba(60, 88, 128, 0.75)",
        hovertemplate="Date=%{x}<br>Value=%{y}",
        legendgroup="Train",
        showlegend=row == 1,
        row=row,
        col=1,
    )
    fig.add_scatter(
        name="Test",
        x=test_dates,
        y=arima_forecast[:, component],
        line_color="rgba(224, 76, 95, 0.75)",
        hovertemplate="Date=%{x}<br>Value=%{y}",
        legendgroup="Test",
        showlegend=row == 1,
        row=row,
        col=1,
    )
    fig.add_vline(x="2022-01-01", line_width=2, line_dash="dash", line_color="#2D425E")
    fig.update_xaxes(showgrid=False, range=("2018-12-01", "2022-12-31"))
    fig.update_yaxes(showgrid=False)

fig.update_traces(line_width=1.5)
fig.update_layout(
    title="CO\u2082 Emission Decomposition with NMF<br>"
    "<span style='font-size: 75%; font-weight: bold;'>"
    "The red lines depict forecast (using ARIMA models) for 2022</span>",
    font_color=FONT_COLOR,
    title_font_size=18,
    title_y=0.94,
    margin_t=130,
    plot_bgcolor=BACKGROUND_COLOR,
    paper_bgcolor=BACKGROUND_COLOR,
    width=840,
    height=740,
    legend=dict(yanchor="bottom", xanchor="right", y=1, x=1, orientation="h", title=""),
)
fig.show()


<p style="
    font-size: 20px;
    font-family: 'JetBrains Mono';
    color: #4A4B52;
    border-bottom: 2px solid #E04C5F;
">
    <b>ARIMA Forecast</b> 📜
</p>

<p style="
    font-size: 16px;
    font-family: 'JetBrains Mono';
    text-align: justify;
    text-justify: inter-word;
">
    In the cases of ARIMA models, predictions look more natural than when we used Random Forest. However, the score on public LB is worse. 
</p>

In [ ]:
test_arima_forecast = (
    pd.DataFrame(
        nmf.inverse_transform(arima_forecast),
        columns=emission_pivot.columns,
        index=pd.MultiIndex.from_arrays(indices, names=("Year", "Week_No")),
    )
    .melt(ignore_index=False, value_name="Emission")
    .reset_index()
    .set_index(test_filled_extended.index)
)

test_arima_forecast.head().style.set_table_styles(DF_STYLE)


In [ ]:
submission_arima = pd.DataFrame(
    {
        "ID_LAT_LON_YEAR_WEEK": test_arima_forecast.index,
        "Emission": test_arima_forecast.Emission,
    }
).set_index("ID_LAT_LON_YEAR_WEEK")

submission_arima.to_csv("submission_arima.csv")
submission_arima.head().style.set_table_styles(DF_STYLE)


# <b> <span style="font-family: 'JetBrains Mono'; color: #4A4B52">5</span> <span style='color: #E04C5F'>|</span> <span style="font-family: 'JetBrains Mono'; color: #4A4B52">Summary</span></b><a class="anchor" id="summary"></a> [↑](#top)

In [ ]:
# NMF approach is the best.
final_submission = submission_nmf.copy()
final_submission.to_csv("submission.csv")
final_submission.head().style.set_table_styles(DF_STYLE)


<p style="
    font-size: 16px;
    font-family: 'JetBrains Mono';
    text-align: justify;
    text-justify: inter-word;
">
    That's probably all in this notebook. I hope you like it, and if yes, then leave an upvote. If you have any questions, let me know in the comments section. Moreover, I encourage you to check my other notebooks, especially these: <a href="https://www.kaggle.com/code/mateuszk013/neural-machine-translation-with-attention"><b>Neural Machine Translation with Attention</b></a> and <a href="https://www.kaggle.com/code/mateuszk013/ml-explainability-shapley-values-shap"><b>ML Explainability - Shapley Values & SHAP</b></a>. These are important to me. Thanks and good luck!
</p>